# SaltySeq - XGBoost Training với Time Series Cross-Validation

**Target**: `is_stress_event` (Binary Classification)

**Tuning Metric**: AUPRC (Average Precision)

**Test Metrics (reporting)**: Accuracy, Precision, Recall, F1 Score, F2 Score, PR-AUC

**Strategy**:
- Dataset: `data/merged_final.csv`
- Train set: từ 2015-01-01 đến 2022-12-31
- Test set: từ 2023-01-01 đến 2025-12-31
- Cross-validation: TimeSeriesSplit với `n_splits=5` trên train set
- Optuna (TPE sampler) cho hyperparameter tuning
- Final model train lại trên toàn bộ train set với best params, `eval_metric='aucpr'` và `scale_pos_weight`

## 1. Import Libraries và Setup

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
import os
from pathlib import Path
warnings.filterwarnings('ignore')

from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix,
    classification_report, accuracy_score, f1_score, precision_score, recall_score,
    average_precision_score, precision_recall_curve, fbeta_score
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import joblib

# Optuna for hyperparameter optimization
try:
    import optuna
    from optuna.samplers import TPESampler
except ImportError:
    import sys
    import subprocess
    print("Optuna not found. Installing optuna...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "optuna"] )
    import optuna
    from optuna.samplers import TPESampler

# Setup plotting
plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
sns.set_style('whitegrid')
sns.set_palette('husl')

# Paths
ROOT_DIR = Path.cwd().resolve()
if (ROOT_DIR / "backend" / "data").exists():
    BACKEND_DIR = ROOT_DIR / "backend"
elif ROOT_DIR.name == "models" and (ROOT_DIR.parent / "data").exists():
    BACKEND_DIR = ROOT_DIR.parent
else:
    BACKEND_DIR = ROOT_DIR

DATA_DIR = BACKEND_DIR / "data"
OUTPUT_DIR = BACKEND_DIR / "models"
REPORT_DIR = BACKEND_DIR / "reports"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration
DATA_FILE = DATA_DIR / "merged_final.csv"
SORTED_DATA_FILE = DATA_DIR / "sorted.csv"
TARGET_COL = 'is_stress_event'
TRAIN_START_DATE = '2015-01-01'
TRAIN_END_DATE = '2022-12-31'
TEST_START_DATE = '2023-01-01'
TEST_END_DATE = '2025-12-31'
N_SPLITS_CV = 5

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## 2. Load và Preprocessing Data

In [7]:
print("=" * 80)
print("  STEP 1: DATA LOADING & PREPROCESSING")
print("=" * 80)

# Load
print(f"\n[1.1] Loading data from: {DATA_FILE}")
df = pd.read_csv(DATA_FILE, parse_dates=['date'])

# Ensure strict temporal ordering: sort by date first, then by location within each date
if 'location_id' in df.columns:
    sort_cols = ['date', 'location_id']
else:
    sort_cols = ['date']
df = df.sort_values(sort_cols, kind='mergesort').reset_index(drop=True)

# Export sorted file for verification
df.to_csv(SORTED_DATA_FILE, index=False)
print(f"  Sorted by: {sort_cols}")
print(f"  Saved sorted data to: {SORTED_DATA_FILE}")

print(f"  Shape: {df.shape}")
print(f"  Date range: {df['date'].min().date()} → {df['date'].max().date()}")

preview_cols = ['date'] + (['location_id'] if 'location_id' in df.columns else [])
print("\n  Preview sorted keys (first 10 rows):")
print(df[preview_cols].head(10).to_string(index=False))

# Check target
if TARGET_COL not in df.columns:
    print(f"\n❌ ERROR: Target column '{TARGET_COL}' not found!")
    print(f"Available columns: {df.columns.tolist()}")
    
    if 'ndvi_zscore' in df.columns:
        print(f"\n[!] Creating '{TARGET_COL}' from ndvi_zscore < -2...")
        df[TARGET_COL] = (df['ndvi_zscore'] < -2.0).astype(int)
    else:
        raise ValueError(f"Cannot create target '{TARGET_COL}'")

# Target distribution
n_pos = df[TARGET_COL].sum()
n_total = len(df)
print(f"\n[1.2] Target distribution:")
print(f"  Positive (stress=1): {n_pos} ({n_pos/n_total*100:.1f}%)")
print(f"  Negative (stress=0): {n_total-n_pos} ({(n_total-n_pos)/n_total*100:.1f}%)")

if n_pos < 10:
    print(f"\n⚠️  WARNING: Very few positive samples! Class imbalance severe.")

  STEP 1: DATA LOADING & PREPROCESSING

[1.1] Loading data from: C:\Users\F15\Downloads\saltyseq-demo\backend\data\merged_final.csv
  Sorted by: ['date', 'location_id']
  Saved sorted data to: C:\Users\F15\Downloads\saltyseq-demo\backend\data\sorted.csv
  Shape: (20090, 70)
  Date range: 2015-01-01 → 2025-12-31

  Preview sorted keys (first 10 rows):
      date  location_id
2015-01-01     BT_BaTri
2015-01-01   BT_BinhDai
2015-01-01 BT_ChauThanh
2015-01-01 BT_GiongTrom
2015-01-01  BT_ThanhPhu
2015-01-02     BT_BaTri
2015-01-02   BT_BinhDai
2015-01-02 BT_ChauThanh
2015-01-02 BT_GiongTrom
2015-01-02  BT_ThanhPhu

[1.2] Target distribution:
  Positive (stress=1): 2171 (10.8%)
  Negative (stress=0): 17919 (89.2%)


In [8]:
# Features to drop (aligned with Feature Engineering V1)
drop_cols = [
    # Not model features / identifiers
    'date',
    'ndvi_source', 'season',
    'ndvi_is_observed', 'lst_is_observed',

    # Targets and target-like columns
    'is_ndvi_anomaly', 'is_salinity_spike', 'is_stress_event',
    'crop_stress_score',

    # Leakage / target-derived features
    'ndvi_zscore',
    'ndvi_monthly_mean', 'ndvi_monthly_std', 'monthly_ndvi_zscore',
    'ndvi_diff', 'ndvi_pct_change',

    # Multicollinearity-prone salinity lags
    'salinity_lag_1', 'salinity_lag_3', 'salinity_lag_7',

    # Duplicate / low-informative precipitation features
    'precip_7d_mm',
    'rain',
    'precip_lag_1', 'precip_lag_3', 'precip_lag_7',

    # Intermediate calculations (if present)
    'salinity_3d_diff',
]

# Drop columns
drop_cols_exist = [c for c in drop_cols if c in df.columns]
print(f"\n[1.3] Dropping {len(drop_cols_exist)} columns:")
print(f"  {', '.join(drop_cols_exist[:12])}...")

X = df.drop(columns=drop_cols_exist, errors='ignore')
y = df[TARGET_COL]
dates = df['date']

# Handle non-numeric
non_numeric = X.select_dtypes(exclude=[np.number]).columns.tolist()
if non_numeric:
    print(f"\n[1.4] Found {len(non_numeric)} non-numeric columns: {non_numeric}")
    print(f"  Dropping them...")
    X = X.drop(columns=non_numeric)

print(f"\n[1.5] Final feature set:")
print(f"  Features: {X.shape[1]} columns")
print(f"  Samples:  {X.shape[0]} rows")
print(f"  Target:   {y.name}")

# Missing values
missing = X.isnull().sum().sum()
if missing > 0:
    print(f"\n[1.6] Handling missing values: {missing} total")
    X = X.ffill().bfill()
    remaining = X.isnull().sum().sum()
    print(f"  After ffill/bfill: {remaining} remaining")
    
    if remaining > 0:
        X = X.fillna(X.median())
        print(f"  After median fill: {X.isnull().sum().sum()} remaining")

# Infinite values
inf_mask = np.isinf(X.select_dtypes(include=[np.number])).any()
if inf_mask.any():
    print(f"\n[1.7] Found infinite values in {inf_mask.sum()} columns")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median())
    print(f"  Replaced with median")

print(f"\n✅ Preprocessing complete!")
print(f"  X: {X.shape}, y: {y.shape}")

print(X.columns.tolist())


[1.3] Dropping 19 columns:
  date, ndvi_source, season, ndvi_is_observed, lst_is_observed, is_salinity_spike, is_stress_event, crop_stress_score, ndvi_zscore, ndvi_diff, ndvi_pct_change, salinity_lag_1...

[1.4] Found 5 non-numeric columns: ['location_id', 'location_name', 'ndvi_interp_method', 'lst_interp_method', 'salinity_source']
  Dropping them...

[1.5] Final feature set:
  Features: 46 columns
  Samples:  20090 rows
  Target:   is_stress_event

[1.6] Handling missing values: 5043 total
  After ffill/bfill: 0 remaining

✅ Preprocessing complete!
  X: (20090, 46), y: (20090,)
['lat', 'lon', 'distance_to_estuary_km', 'ndvi', 'lst', 'ndvi_gap_days', 'temp_max', 'temp_min', 'temp_mean', 'precipitation', 'et0', 'radiation', 'wind_max', 'soil_moisture_surface', 'soil_moisture_deep', 'soil_temp', 'salinity_psu', 'day_of_year', 'month', 'week', 'is_dry_season', 'month_sin', 'month_cos', 'temp_7d_avg', 'ndvi_7d_avg', 'precip_7d_sum', 'salinity_7d_avg', 'salinity_7d_avg_lag1', 'lst_7d_avg

## 3. Train-Test Split (Time-based)

In [9]:
print("\n" + "=" * 80)
print("  STEP 2: TIME-BASED TRAIN-TEST SPLIT")
print("=" * 80)

# Ensure sorting before split
split_df = pd.DataFrame(index=X.index)
split_df['date'] = pd.to_datetime(dates)

train_mask = (split_df['date'] >= pd.Timestamp(TRAIN_START_DATE)) & (split_df['date'] <= pd.Timestamp(TRAIN_END_DATE))
test_mask = (split_df['date'] >= pd.Timestamp(TEST_START_DATE)) & (split_df['date'] <= pd.Timestamp(TEST_END_DATE))

if not train_mask.any():
    raise ValueError("Train mask is empty. Check TRAIN_START_DATE/TRAIN_END_DATE.")
if not test_mask.any():
    raise ValueError("Test mask is empty. Check TEST_START_DATE/TEST_END_DATE.")

# Build splits
X_train = X.loc[train_mask]
y_train = y.loc[train_mask]
dates_train = dates.loc[train_mask]

X_test = X.loc[test_mask]
y_test = y.loc[test_mask]
dates_test = dates.loc[test_mask]

# -------- Overlap checks --------
train_idx_set = set(X_train.index)
test_idx_set = set(X_test.index)
index_overlap = train_idx_set.intersection(test_idx_set)
if len(index_overlap) > 0:
    raise ValueError(f"Index overlap detected: {len(index_overlap)} rows appear in both train and test.")

train_date_set = set(pd.to_datetime(dates_train).dt.date)
test_date_set = set(pd.to_datetime(dates_test).dt.date)
date_overlap = train_date_set.intersection(test_date_set)
if len(date_overlap) > 0:
    raise ValueError(f"Date overlap detected: {len(date_overlap)} calendar days overlap.")

if dates_train.max() >= dates_test.min():
    raise ValueError("Temporal boundary overlap detected: max(train_date) >= min(test_date).")

panel_overlap_count = 0
if 'location_id' in df.columns:
    train_keys = df.loc[train_mask, ['location_id', 'date']].drop_duplicates()
    test_keys = df.loc[test_mask, ['location_id', 'date']].drop_duplicates()
    panel_overlap_count = int(
        train_keys.merge(test_keys, on=['location_id', 'date'], how='inner').shape[0]
    )
    if panel_overlap_count > 0:
        raise ValueError(
            f"Panel-key overlap detected: {panel_overlap_count} (location_id, date) keys overlap."
        )

print(f"\n[2.1] Split configuration:")
print(f"  Train: {TRAIN_START_DATE} → {TRAIN_END_DATE}")
print(f"    Samples: {len(X_train)}")
print(f"    Date range: {dates_train.min().date()} → {dates_train.max().date()}")
print(f"    Unique dates: {dates_train.dt.normalize().nunique()}")
print(f"    Positives:  {y_train.sum()} / {len(y_train)} ({y_train.sum()/len(y_train)*100:.1f}%)")

print(f"\n  Test:  {TEST_START_DATE} → {TEST_END_DATE}")
print(f"    Samples: {len(X_test)}")
print(f"    Date range: {dates_test.min().date()} → {dates_test.max().date()}")
print(f"    Unique dates: {dates_test.dt.normalize().nunique()}")
print(f"    Positives:  {y_test.sum()} / {len(y_test)} ({y_test.sum()/len(y_test)*100:.1f}%)")

if 'location_id' in df.columns:
    n_locs = int(df['location_id'].nunique())
    test_days = int(dates_test.dt.normalize().nunique())
    expected_test_rows = n_locs * test_days
    print(f"\n[2.2] Panel-data explanation:")
    print(f"  Locations: {n_locs}")
    print(f"  Test days: {test_days}")
    print(f"  Expected rows if full panel = locations × days = {n_locs} × {test_days} = {expected_test_rows}")
    print("  Vì dataset là panel theo nhiều location mỗi ngày, test set có thể > 4000 mẫu là bình thường.")

print(f"\n[2.3] Overlap checks passed:")
print(f"  Index overlap rows: {len(index_overlap)}")
print(f"  Date overlap days:  {len(date_overlap)}")
if 'location_id' in df.columns:
    print(f"  Panel-key overlap:  {panel_overlap_count}")
print("✅ No overlap between train and test.")


  STEP 2: TIME-BASED TRAIN-TEST SPLIT

[2.1] Split configuration:
  Train: 2015-01-01 → 2022-12-31
    Samples: 14610
    Date range: 2015-01-01 → 2022-12-31
    Unique dates: 2922
    Positives:  1469 / 14610 (10.1%)

  Test:  2023-01-01 → 2025-12-31
    Samples: 5480
    Date range: 2023-01-01 → 2025-12-31
    Unique dates: 1096
    Positives:  702 / 5480 (12.8%)

[2.2] Panel-data explanation:
  Locations: 5
  Test days: 1096
  Expected rows if full panel = locations × days = 5 × 1096 = 5480
  Vì dataset là panel theo nhiều location mỗi ngày, test set có thể > 4000 mẫu là bình thường.

[2.3] Overlap checks passed:
  Index overlap rows: 0
  Date overlap days:  0
  Panel-key overlap:  0
✅ No overlap between train and test.


## 4. Feature Scaling

In [5]:
# print("\n" + "=" * 80)
# print("  STEP 3: FEATURE SCALING")
# print("=" * 80)

# scaler = StandardScaler()

# print(f"\n[3.1] Fitting StandardScaler on train set...")
# X_train_scaled = pd.DataFrame(
#     scaler.fit_transform(X_train),
#     columns=X_train.columns,
#     index=X_train.index
# )

# print(f"[3.2] Transforming test set...")
# X_test_scaled = pd.DataFrame(
#     scaler.transform(X_test),
#     columns=X_test.columns,
#     index=X_test.index
# )

# print(f"\n✅ Feature scaling complete!")
# print(f"  Train: mean={X_train_scaled.mean().mean():.4f}, std={X_train_scaled.std().mean():.4f}")
# print(f"  Test:  mean={X_test_scaled.mean().mean():.4f}, std={X_test_scaled.std().mean():.4f}")

## 5. Hyperparameter Tuning với Optuna

In [6]:
# print("\n" + "=" * 80)
# print("  STEP 4: HYPERPARAMETER TUNING (OPTUNA)")
# print("=" * 80)

# # TimeSeriesSplit
# tscv = TimeSeriesSplit(n_splits=N_SPLITS_CV)
# print(f"\n[4.1] TimeSeriesSplit configuration:")
# print(f"  n_splits: {N_SPLITS_CV}")
# print(f"  Each fold: ~{len(X_train) // (N_SPLITS_CV + 1)} samples for train/val")

# # Visualize splits
# print(f"\n[4.2] Time series split visualization:")
# for i, (train_idx, val_idx) in enumerate(tscv.split(X_train)):
#     print(f"  Fold {i+1}: Train[0:{train_idx[-1]+1:5d}] → Val[{val_idx[0]:5d}:{val_idx[-1]+1:5d}]")

# # Class imbalance handling
# n_neg = (y_train == 0).sum()
# n_pos = (y_train == 1).sum()
# scale_pos_weight = n_neg / n_pos if n_pos > 0 else 1
# print(f"\n[4.3] Class imbalance handling:")
# print(f"  Negatives: {n_neg}, Positives: {n_pos}")
# print(f"  scale_pos_weight: {scale_pos_weight:.2f}")

In [7]:
# # Optuna search space configuration
# N_TRIALS = 100

# print(f"\n[4.4] Optuna search space:")
# print("  n_estimators      : [150, 900] (int)")
# print("  max_depth         : [3, 10] (int)")
# print("  learning_rate     : [0.005, 0.2] (log)")
# print("  subsample         : [0.6, 1.0]")
# print("  colsample_bytree  : [0.6, 1.0]")
# print("  min_child_weight  : [1, 12] (int)")
# print("  gamma             : [0.0, 5.0]")
# print("  reg_alpha         : [1e-8, 10.0] (log)")
# print("  reg_lambda        : [1e-3, 20.0] (log)")
# print(f"  n_trials          : {N_TRIALS}")

In [8]:
# def objective(trial):
#     params = {
#         'n_estimators': trial.suggest_int('n_estimators', 150, 900),
#         'max_depth': trial.suggest_int('max_depth', 3, 10),
#         'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 12),
#         'gamma': trial.suggest_float('gamma', 0.0, 5.0),
#         'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
#         'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 20.0, log=True),
#         'objective': 'binary:logistic',
#         'eval_metric': 'aucpr',
#         'scale_pos_weight': scale_pos_weight,
#         'random_state': 42,
#         'n_jobs': -1,
#         'verbosity': 0,
#     }

#     fold_scores = []
#     for train_idx, val_idx in tscv.split(X_train_scaled):
#         X_tr = X_train_scaled.iloc[train_idx]
#         y_tr = y_train.iloc[train_idx]
#         X_val = X_train_scaled.iloc[val_idx]
#         y_val = y_train.iloc[val_idx]

#         model = XGBClassifier(**params)
#         model.fit(X_tr, y_tr)

#         y_val_proba = model.predict_proba(X_val)[:, 1]
#         auprc = average_precision_score(y_val, y_val_proba)
#         fold_scores.append(auprc)

#     mean_auprc = float(np.mean(fold_scores))
#     std_auprc = float(np.std(fold_scores))
#     trial.set_user_attr('fold_scores', fold_scores)
#     trial.set_user_attr('std_auprc', std_auprc)
#     return mean_auprc

# print(f"\n[4.5] Starting Optuna optimization...")
# print(f"  Objective: maximize mean CV AUPRC")
# print(f"  eval_metric inside model: aucpr")
# print(f"  scale_pos_weight: {scale_pos_weight:.4f}")

# optuna.logging.set_verbosity(optuna.logging.WARNING)
# sampler = TPESampler(seed=42)
# study = optuna.create_study(direction='maximize', sampler=sampler)
# study.optimize(objective, n_trials=N_TRIALS, n_jobs=1, show_progress_bar=False)

# print(f"\n✅ Optuna optimization complete!")

In [9]:
# best_params = study.best_params.copy()
# best_cv_score = float(study.best_value)

# print(f"\n[4.6] Best hyperparameters (Optuna):")
# for param, value in best_params.items():
#     print(f"  {param:20s}: {value}")

# print(f"\n[4.7] Best cross-validation score (AUPRC): {best_cv_score:.4f}")

# # Build results DataFrame compatible with downstream visualizations
# completed_trials = [t for t in study.trials if t.value is not None and t.state == optuna.trial.TrialState.COMPLETE]
# rows = []
# for t in completed_trials:
#     row = {f"param_{k}": v for k, v in t.params.items()}
#     row['mean_test_score'] = float(t.value)
#     row['std_test_score'] = float(t.user_attrs.get('std_auprc', np.nan))
#     row['params'] = t.params
#     rows.append(row)

# results_df = pd.DataFrame(rows)
# if results_df.empty:
#     raise ValueError("No completed Optuna trials were found.")

# results_df = results_df.sort_values('mean_test_score', ascending=False).reset_index(drop=True)
# results_df['rank_test_score'] = (
#     results_df['mean_test_score'].rank(method='min', ascending=False).astype(int)
# )

# print(f"\n[4.8] Top 10 hyperparameter combinations:")
# print(f"  {'Rank':<6} {'Mean AUPRC':<12} {'Std AUPRC':<12} {'Params'}")
# print(f"  {'-'*6} {'-'*12} {'-'*12} {'-'*50}")

# for _, row in results_df.head(10).iterrows():
#     print(f"  {row['rank_test_score']:<6.0f} "
#           f"{row['mean_test_score']:<12.4f} "
#           f"{row['std_test_score']:<12.4f} "
#           f"{str(row['params'])[:50]}")

# # Save Optuna trials
# results_file = OUTPUT_DIR / "optuna_trials.csv"
# results_df.to_csv(results_file, index=False)
# print(f"\n[4.9] Saved full results to: {results_file}")

## 6. Visualize Tuning Results (XGBoost)

Sử dụng 2 biểu đồ giống với additional models để đảm bảo giao diện đồng nhất:
- Optimization History (Trial AUPRC và Best-so-far AUPRC)
- Hyperparameter Importance

In [10]:
# print("\n" + "=" * 80)
# print("  STEP 5: XGBOOST OPTUNA VISUALIZATION (2 PLOTS)")
# print("=" * 80)

# if 'study' not in globals():
#     raise ValueError("'study' chưa tồn tại. Hãy chạy xong bước Optuna tuning trước.")

# complete_trials = [
#     t for t in study.trials
#     if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None
# ]
# if len(complete_trials) == 0:
#     raise ValueError("Study chưa có completed trial nào. Hãy chạy cell optimize trước.")

# fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# # Plot 1: Optimization history (same style as additional models)
# trial_plot_df = pd.DataFrame({
#     "trial": [t.number for t in complete_trials],
#     "auprc": [t.value for t in complete_trials],
# }).sort_values("trial")

# axes[0].plot(
#     trial_plot_df["trial"],
#     trial_plot_df["auprc"],
#     marker="o",
#     linewidth=1.5,
#     alpha=0.8,
#     label="Trial AUPRC",
# )
# axes[0].plot(
#     trial_plot_df["trial"],
#     trial_plot_df["auprc"].cummax(),
#     linewidth=2.5,
#     color="darkorange",
#     label="Best-so-far AUPRC",
# )
# axes[0].set_title("XGBoost - Optimization History")
# axes[0].set_xlabel("Trial")
# axes[0].set_ylabel("Mean CV AUPRC")
# axes[0].legend()
# axes[0].grid(alpha=0.3)

# # Plot 2: Hyperparameter importance (same style as additional models)
# try:
#     param_importance = optuna.importance.get_param_importances(study)
# except Exception:
#     param_importance = {}

# if len(param_importance) > 0:
#     imp_names = list(param_importance.keys())[::-1]
#     imp_values = list(param_importance.values())[::-1]
#     axes[1].barh(imp_names, imp_values, color="steelblue", alpha=0.85)
#     axes[1].set_title("XGBoost - Hyperparameter Importance")
#     axes[1].set_xlabel("Importance")
#     axes[1].grid(axis="x", alpha=0.3)
# else:
#     axes[1].text(
#         0.5, 0.5,
#         "Could not compute parameter importance\n(skip if dependencies are missing)",
#         ha="center",
#         va="center",
#     )
#     axes[1].set_axis_off()

# plt.tight_layout()
# xgb_tuning_chart_path = REPORT_DIR / "optuna_tuning_xgboost.png"
# plt.savefig(xgb_tuning_chart_path, dpi=300, bbox_inches="tight")
# plt.show()

# print(f"\n[5.1] Saved tuning chart: {xgb_tuning_chart_path}")
# print("✅ XGBoost Optuna visualization complete!")

In [11]:
# print("\n" + "=" * 80)
# print("  STEP 5B: VISUALIZATION NOTE")
# print("=" * 80)

# print("XGBoost visualization đã được chuẩn hóa về 2 plot giống additional models.")
# print("Không tạo thêm advanced plot để giữ giao diện đồng nhất.")

# if 'xgb_tuning_chart_path' in globals():
#     print(f"Saved file: {xgb_tuning_chart_path}")
# else:
#     print("Hãy chạy STEP 5 trước để tạo biểu đồ.")

## 7. Retrain Best Model và Final Evaluation on Test Set

In [12]:
# print("\n" + "=" * 80)
# print("  STEP 6: RETRAIN BEST MODEL + FINAL EVALUATION ON TEST SET")
# print("=" * 80)

# # Retrain final model on full training set with best hyperparameters from Optuna
# final_best_params = best_params.copy()
# print(f"\n[6.0] Retraining final model with best hyperparameters...")
# print(f"  Best params: {final_best_params}")
# print(f"  eval_metric: aucpr")
# print(f"  scale_pos_weight: {scale_pos_weight:.4f}")

# # Final model on full train
# final_model = XGBClassifier(
#     **final_best_params,
#     objective='binary:logistic',
#     eval_metric='aucpr',
#     scale_pos_weight=scale_pos_weight,
#     random_state=42,
#     n_jobs=-1,
#     verbosity=0
# )
# final_model.fit(X_train_scaled, y_train)
# print("  ✅ Final model retrained on full train set")

# # Predictions (fixed threshold = 0.5, no threshold tuning)
# best_threshold = 0.5
# y_pred_proba = final_model.predict_proba(X_test_scaled)[:, 1]
# y_pred = (y_pred_proba >= best_threshold).astype(int)

# # PR-AUC and classification metrics
# test_pr_auc = average_precision_score(y_test, y_pred_proba)
# print(f"\n[6.1] 🎯 TEST SET PR-AUC: {test_pr_auc:.4f}")

# print(f"\n[6.2] Classification Report (threshold={best_threshold:.4f}):")
# print(classification_report(y_test, y_pred, target_names=['No Stress', 'Stress'], zero_division=0))

# # Confusion matrix
# cm = confusion_matrix(y_test, y_pred)
# print(f"\n[6.3] Confusion Matrix:")
# print(f"                 Predicted")
# print(f"                No Stress  Stress")
# print(f"  Actual No Stress  {cm[0,0]:6d}  {cm[0,1]:6d}")
# print(f"         Stress     {cm[1,0]:6d}  {cm[1,1]:6d}")

# # Additional metrics
# precision = precision_score(y_test, y_pred, zero_division=0)
# recall = recall_score(y_test, y_pred, zero_division=0)
# f1 = f1_score(y_test, y_pred, zero_division=0)
# f2 = fbeta_score(y_test, y_pred, beta=2, zero_division=0)

# print(f"\n[6.4] Summary Metrics:")
# print(f"  PR-AUC:    {test_pr_auc:.4f}")
# print(f"  Threshold: {best_threshold:.4f}")
# print(f"  Precision: {precision:.4f} (of predicted stress, % actually stress)")
# print(f"  Recall:    {recall:.4f} (of actual stress, % detected)")
# print(f"  F1-Score:  {f1:.4f}")
# print(f"  F2-Score:  {f2:.4f} (recall-weighted)")

In [13]:
# # Feature importance using gain
# print(f"\n[6.5] Top 15 Most Important Features (Gain):")

# gain_scores = final_model.get_booster().get_score(importance_type='gain')
# feature_imp = pd.DataFrame({'feature': X_train.columns})
# feature_imp['importance'] = feature_imp['feature'].map(gain_scores).fillna(0.0)
# feature_imp = feature_imp.sort_values('importance', ascending=False)

# print(f"  {'Rank':<6} {'Feature':<35} {'Gain':<12} {'Bar'}")
# print(f"  {'-'*6} {'-'*35} {'-'*12} {'-'*30}")
# for i, row in feature_imp.head(15).iterrows():
#     bar = '█' * int((row['importance'] / feature_imp['importance'].max()) * 30) if feature_imp['importance'].max() > 0 else ''
#     rank = feature_imp.index.get_loc(i) + 1
#     print(f"  {rank:<6} {row['feature']:<35} {row['importance']:<12.4f} {bar}")

# # Save feature importance
# feat_imp_file = OUTPUT_DIR / "feature_importance.csv"
# feature_imp.to_csv(feat_imp_file, index=False)
# print(f"\n[6.6] Saved feature importance: {feat_imp_file}")

In [14]:
# # Plot PR curve and feature importance
# print(f"\n[6.7] Plotting PR curve and feature importance...")

# fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# # PR curve
# ax = axes[0]
# curve_precision, curve_recall, _ = precision_recall_curve(y_test, y_pred_proba)
# positive_rate = y_test.mean()

# ax.plot(curve_recall, curve_precision, linewidth=3,
#         label=f'XGBoost (PR-AUC = {test_pr_auc:.4f})', color='darkorange')
# ax.axhline(y=positive_rate, color='k', linestyle='--', linewidth=2,
#            label=f'Baseline (Positive rate = {positive_rate:.4f})')
# ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
# ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
# ax.set_title('PR Curve - Test Set', fontsize=14, fontweight='bold')
# ax.legend(loc='lower left', fontsize=11)
# ax.grid(alpha=0.3)

# # Feature importance plot
# ax = axes[1]
# top_15 = feature_imp.head(15)
# ax.barh(range(15), top_15['importance'], color='steelblue', alpha=0.8)
# ax.set_yticks(range(15))
# ax.set_yticklabels(top_15['feature'])
# ax.set_xlabel('Gain', fontsize=12, fontweight='bold')
# ax.set_title('Top 15 Feature Importance (Gain)', fontsize=14, fontweight='bold')
# ax.invert_yaxis()
# ax.grid(axis='x', alpha=0.3)

# plt.tight_layout()

# eval_fig_path = REPORT_DIR / "final_evaluation.png"
# plt.savefig(eval_fig_path, dpi=300, bbox_inches='tight')
# print(f"[6.8] Saved evaluation figure: {eval_fig_path}")

# plt.show()

# # Save metrics
# metrics = {
#     'cv_best_auprc': float(best_cv_score),
#     'tuner': 'optuna',
#     'tuning_objective': 'aucpr',
#     'eval_metric': 'aucpr',
#     'scale_pos_weight': float(scale_pos_weight),
#     'test_pr_auc': float(test_pr_auc),
#     'decision_threshold': float(best_threshold),
#     'precision': float(precision),
#     'recall': float(recall),
#     'f1_score': float(f1),
#     'f2_score': float(f2),
#     'confusion_matrix': cm.tolist(),
#     'n_test_samples': len(y_test),
#     'n_test_positive': int(y_test.sum()),
# }

# metrics_file = OUTPUT_DIR / "test_metrics.json"
# with open(metrics_file, 'w') as f:
#     json.dump(metrics, f, indent=2)
# print(f"[6.9] Saved metrics: {metrics_file}")

# print(f"\n✅ Final evaluation complete!")

## 8. Save Model và Scaler

In [15]:
# print("\n" + "=" * 80)
# print("  SAVING TRAINED MODEL")
# print("=" * 80)

# model_file = OUTPUT_DIR / "xgboost_best_model.pkl"
# scaler_file = OUTPUT_DIR / "feature_scaler.pkl"

# joblib.dump(final_model, model_file)
# joblib.dump(scaler, scaler_file)

# print(f"\n[✓] Saved best model:     {model_file}")
# print(f"[✓] Saved scaler:         {scaler_file}")

## 9. Summary

In [16]:
# print("\n" + "=" * 80)
# print("  🎉 TRAINING PIPELINE COMPLETE!")
# print("=" * 80)

# print(f"\n📊 Final Results:")
# print(f"  • Best CV score (AUPRC):  {best_cv_score:.4f}")
# print(f"  • Test PR-AUC:            {test_pr_auc:.4f}")
# print(f"  • Test F2-Score:          {f2:.4f}")
# print(f"  • Decision threshold:     {best_threshold:.4f}")
# print(f"  • Top 3 features (gain):")
# for idx, row in feature_imp.head(3).iterrows():
#     rank = feature_imp.index.get_loc(idx) + 1
#     print(f"      {rank}. {row['feature']:30s} ({row['importance']:.4f})")

# print(f"\n🧭 Training setup:")
# print(f"  • Data file:             {DATA_FILE}")
# print(f"  • Train period:          {TRAIN_START_DATE} → {TRAIN_END_DATE}")
# print(f"  • Test period:           {TEST_START_DATE} → {TEST_END_DATE}")
# print(f"  • CV splits:             {N_SPLITS_CV}")
# print(f"  • Tuner:                 Optuna (TPE)")
# print(f"  • tuning objective:      AUPRC")
# print(f"  • eval_metric:           aucpr")
# print(f"  • scale_pos_weight:      {scale_pos_weight:.4f}")

# print(f"\n📁 Output files:")
# print(f"  • Model:                {OUTPUT_DIR / 'xgboost_best_model.pkl'}")
# print(f"  • Scaler:               {OUTPUT_DIR / 'feature_scaler.pkl'}")
# print(f"  • Optuna Trials CSV:    {OUTPUT_DIR / 'optuna_trials.csv'}")
# print(f"  • Feature Importance:   {OUTPUT_DIR / 'feature_importance.csv'}")
# print(f"  • Test Metrics:         {OUTPUT_DIR / 'test_metrics.json'}")
# print(f"  • XGBoost Tuning Plot:  {REPORT_DIR / 'optuna_tuning_xgboost.png'}")
# print(f"  • Final Evaluation:     {REPORT_DIR / 'final_evaluation.png'}")

# print(f"\n" + "=" * 80)

## 10. Multi-Model Optuna Tuning (RandomForest, SVM, Logistic Regression)

Áp dụng cùng pipeline time-based split + TimeSeriesSplit để tune thêm 3 mô hình: RandomForest, SVM, Logistic Regression với objective là **mean CV AUPRC**.

In [17]:
# print("\n" + "=" * 80)
# print("  STEP 7: MULTI-MODEL OPTUNA TUNING (AUPRC)")
# print("=" * 80)

# MULTI_MODEL_N_TRIALS = 100
# MULTI_MODEL_RANDOM_STATE = 42
# multi_model_names = ["RandomForest", "SVM", "LogisticRegression"]

# multi_model_studies = {}
# multi_model_best_params = {}
# multi_model_cv_aucpr = {}
# multi_model_tuning_charts = {}


# def suggest_params_for_model(trial, model_name):
#     if model_name == "RandomForest":
#         return {
#             "n_estimators": trial.suggest_int("n_estimators", 150, 700),
#             "max_depth": trial.suggest_categorical("max_depth", [None, 4, 6, 8, 12, 16, 24]),
#             "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
#             "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
#             "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
#             "class_weight": trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"]),
#         }
#     if model_name == "SVM":
#         kernel = trial.suggest_categorical("kernel", ["rbf", "linear"])
#         gamma = "scale" if kernel == "linear" else trial.suggest_float("gamma", 1e-4, 1e-1, log=True)
#         return {
#             "C": trial.suggest_float("C", 1e-3, 100.0, log=True),
#             "kernel": kernel,
#             "gamma": gamma,
#             "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
#         }
#     if model_name == "LogisticRegression":
#         return {
#             "C": trial.suggest_float("C", 1e-3, 100.0, log=True),
#             "solver": trial.suggest_categorical("solver", ["lbfgs", "liblinear"]),
#             "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
#         }
#     raise ValueError(f"Unsupported model: {model_name}")


# def build_model(model_name, params):
#     if model_name == "RandomForest":
#         return RandomForestClassifier(
#             n_estimators=int(params["n_estimators"]),
#             max_depth=params["max_depth"],
#             min_samples_split=int(params["min_samples_split"]),
#             min_samples_leaf=int(params["min_samples_leaf"]),
#             max_features=params["max_features"],
#             class_weight=params["class_weight"],
#             n_jobs=-1,
#             random_state=MULTI_MODEL_RANDOM_STATE,
#         )
#     if model_name == "SVM":
#         return SVC(
#             C=float(params["C"]),
#             kernel=params["kernel"],
#             gamma=params.get("gamma", "scale"),
#             class_weight=params["class_weight"],
#             probability=True,
#             random_state=MULTI_MODEL_RANDOM_STATE,
#         )
#     if model_name == "LogisticRegression":
#         return LogisticRegression(
#             C=float(params["C"]),
#             solver=params["solver"],
#             class_weight=params["class_weight"],
#             max_iter=5000,
#             random_state=MULTI_MODEL_RANDOM_STATE,
#         )
#     raise ValueError(f"Unsupported model: {model_name}")


# def objective_multimodel(trial, model_name):
#     params = suggest_params_for_model(trial, model_name)
#     fold_scores = []

#     for train_idx, val_idx in tscv.split(X_train_scaled):
#         X_tr = X_train_scaled.iloc[train_idx]
#         y_tr = y_train.iloc[train_idx]
#         X_val = X_train_scaled.iloc[val_idx]
#         y_val = y_train.iloc[val_idx]

#         model = build_model(model_name, params)
#         model.fit(X_tr, y_tr)
#         y_val_proba = model.predict_proba(X_val)[:, 1]

#         fold_auprc = average_precision_score(y_val, y_val_proba)
#         fold_scores.append(fold_auprc)

#     mean_auprc = float(np.mean(fold_scores))
#     std_auprc = float(np.std(fold_scores))
#     trial.set_user_attr("std_auprc", std_auprc)
#     trial.set_user_attr("fold_scores", fold_scores)
#     return mean_auprc


# for i, model_name in enumerate(multi_model_names, start=1):
#     print(f"\n[7.{i}] Tuning {model_name}...")
#     sampler = TPESampler(seed=MULTI_MODEL_RANDOM_STATE)
#     study_m = optuna.create_study(direction="maximize", sampler=sampler, study_name=f"{model_name}_aucpr")
#     study_m.optimize(
#         lambda trial, m=model_name: objective_multimodel(trial, m),
#         n_trials=MULTI_MODEL_N_TRIALS,
#         n_jobs=1,
#         show_progress_bar=False,
#     )

#     multi_model_studies[model_name] = study_m
#     multi_model_best_params[model_name] = study_m.best_params.copy()
#     multi_model_cv_aucpr[model_name] = float(study_m.best_value)

#     print(f"  Best CV AUPRC: {study_m.best_value:.4f}")
#     print(f"  Best params: {study_m.best_params}")

#     slug = model_name.lower().replace(" ", "_")
#     complete_trials = [
#         t for t in study_m.trials
#         if t.state == optuna.trial.TrialState.COMPLETE and t.value is not None
#     ]
#     trial_rows = []
#     for t in complete_trials:
#         row = {"trial": int(t.number), "value": float(t.value), "std_auprc": float(t.user_attrs.get("std_auprc", np.nan))}
#         row.update({f"param_{k}": v for k, v in t.params.items()})
#         trial_rows.append(row)

#     trials_df_m = pd.DataFrame(trial_rows).sort_values("value", ascending=False).reset_index(drop=True)
#     trials_csv_path = OUTPUT_DIR / f"optuna_trials_{slug}.csv"
#     trials_df_m.to_csv(trials_csv_path, index=False)

#     fig, axes = plt.subplots(1, 2, figsize=(14, 5))
#     trial_plot_df = pd.DataFrame({
#         "trial": [t.number for t in complete_trials],
#         "auprc": [t.value for t in complete_trials],
#     }).sort_values("trial")

#     if not trial_plot_df.empty:
#         axes[0].plot(trial_plot_df["trial"], trial_plot_df["auprc"], marker="o", linewidth=1.5, alpha=0.8, label="Trial AUPRC")
#         axes[0].plot(
#             trial_plot_df["trial"],
#             trial_plot_df["auprc"].cummax(),
#             linewidth=2.5,
#             color="darkorange",
#             label="Best-so-far AUPRC",
#         )
#         axes[0].set_title(f"{model_name} - Optimization History")
#         axes[0].set_xlabel("Trial")
#         axes[0].set_ylabel("Mean CV AUPRC")
#         axes[0].legend()
#         axes[0].grid(alpha=0.3)
#     else:
#         axes[0].text(0.5, 0.5, "No completed trials", ha="center", va="center")
#         axes[0].set_axis_off()

#     try:
#         param_importance = optuna.importance.get_param_importances(study_m)
#     except Exception:
#         param_importance = {}

#     if len(param_importance) > 0:
#         imp_names = list(param_importance.keys())[::-1]
#         imp_values = list(param_importance.values())[::-1]
#         axes[1].barh(imp_names, imp_values, color="steelblue", alpha=0.85)
#         axes[1].set_title(f"{model_name} - Hyperparameter Importance")
#         axes[1].set_xlabel("Importance")
#         axes[1].grid(axis="x", alpha=0.3)
#     else:
#         axes[1].text(
#             0.5, 0.5,
#             "Could not compute parameter importance\n(skip if dependencies are missing)",
#             ha="center",
#             va="center",
#         )
#         axes[1].set_axis_off()

#     plt.tight_layout()
#     tuning_chart_path = REPORT_DIR / f"optuna_tuning_{slug}.png"
#     plt.savefig(tuning_chart_path, dpi=300, bbox_inches="tight")
#     plt.close(fig)

#     multi_model_tuning_charts[model_name] = tuning_chart_path
#     print(f"  Saved trials: {trials_csv_path}")
#     print(f"  Saved tuning chart: {tuning_chart_path}")

# print("\n✅ Multi-model Optuna tuning complete!")

In [18]:
# print("\n" + "=" * 80)
# print("  STEP 8: TRAIN FINAL MODELS + TEST EVALUATION")
# print("=" * 80)

# if 'multi_model_best_params' not in globals() or len(multi_model_best_params) == 0:
#     raise ValueError("Không tìm thấy kết quả tuning multi-model. Hãy chạy STEP 7 trước.")

# if 'multi_model_cv_aucpr' in globals():
#     cv_metric_dict = multi_model_cv_aucpr
# elif 'multi_model_cv_f1' in globals():
#     cv_metric_dict = globals()['multi_model_cv_f1']
# else:
#     raise ValueError("Không tìm thấy điểm CV từ multi-model tuning. Hãy chạy lại STEP 7.")


# def evaluate_binary_metrics(y_true, y_pred):
#     return {
#         "Accuracy": float(accuracy_score(y_true, y_pred)),
#         "Precision": float(precision_score(y_true, y_pred, zero_division=0)),
#         "Recall": float(recall_score(y_true, y_pred, zero_division=0)),
#         "F1 Score": float(f1_score(y_true, y_pred, zero_division=0)),
#         "F2 Score": float(fbeta_score(y_true, y_pred, beta=2, zero_division=0)),
#     }


# comparison_rows = []
# final_models = {}

# # 1) Existing XGBoost final model
# if 'final_model' not in globals():
#     raise ValueError("Không tìm thấy final_model (XGBoost). Hãy chạy STEP 6 trước.")

# if 'y_pred' not in globals():
#     y_pred = final_model.predict(X_test_scaled)

# xgb_metrics = evaluate_binary_metrics(y_test, y_pred)
# comparison_rows.append({
#     "Model": "XGBoost",
#     "CV AUPRC (Optuna)": float(best_cv_score),
#     **xgb_metrics,
# })
# final_models["XGBoost"] = final_model

# # 2) Train and evaluate additional models
# for model_name in multi_model_names:
#     params = multi_model_best_params[model_name]
#     model = build_model(model_name, params)
#     model.fit(X_train_scaled, y_train)
#     y_pred_m = model.predict(X_test_scaled)

#     metrics_m = evaluate_binary_metrics(y_test, y_pred_m)
#     comparison_rows.append({
#         "Model": model_name,
#         "CV AUPRC (Optuna)": float(cv_metric_dict[model_name]),
#         **metrics_m,
#     })

#     slug = model_name.lower().replace(" ", "_")
#     model_path = OUTPUT_DIR / f"{slug}_best_model.pkl"
#     joblib.dump(model, model_path)
#     final_models[model_name] = model

#     print(f"  [{model_name}] saved model: {model_path}")
#     print(f"  [{model_name}] test F1: {metrics_m['F1 Score']:.4f}")

# comparison_df = pd.DataFrame(comparison_rows)
# comparison_df = comparison_df[[
#     "Model", "CV AUPRC (Optuna)", "Accuracy", "Precision", "Recall", "F1 Score", "F2 Score"
# ]]
# comparison_df = comparison_df.sort_values("F1 Score", ascending=False).reset_index(drop=True)

# comparison_file = REPORT_DIR / "final_model_comparison.csv"
# comparison_df.to_csv(comparison_file, index=False)
# print(f"\n✅ Saved comparison table: {comparison_file}")
# print("✅ Final models trained and evaluated on test set.")

In [19]:
# print("\n" + "=" * 80)
# print("  FINAL BENCHMARK TABLE (TEST SET)")
# print("=" * 80)

# if 'comparison_df' not in globals() or comparison_df.empty:
#     raise ValueError("comparison_df chưa có dữ liệu. Hãy chạy STEP 8 trước.")

# print(comparison_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# print("\nBest model by Test F1:")
# best_row = comparison_df.iloc[0]
# print(f"  {best_row['Model']} | F1={best_row['F1 Score']:.4f} | F2={best_row['F2 Score']:.4f}")

## 11. PrefixSpan + XGBoost (lookback 7 ngày theo location)

Bổ sung feature tuần tự từ PrefixSpan với ràng buộc panel-data theo `location_id`:
- Mỗi mẫu dùng lịch sử **7 ngày trước đó** tại đúng location tương ứng.
- Không cắt ngang giữa các location/trạm.
- Chọn **Top-K pattern = 10** từ train set rồi transform sang test set.

In [20]:
# print("\n" + "=" * 80)
# print("  STEP 11.1: PREFIXSPAN SETUP")
# print("=" * 80)

# # PrefixSpan import (auto-install if needed)
# try:
#     from prefixspan import PrefixSpan
# except ImportError:
#     import sys
#     import subprocess
#     print("PrefixSpan not found. Installing prefixspan...")
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "prefixspan"])
#     from prefixspan import PrefixSpan

# # Core configuration
# LOOKBACK_DAYS_PS = 3
# TOP_K_POS_PATTERNS_PS = 10
# TOP_K_NEG_PATTERNS_PS = 20
# TOP_K_PATTERNS_PS = TOP_K_POS_PATTERNS_PS + TOP_K_NEG_PATTERNS_PS
# PREFIXSPAN_MAXLEN_PS = min(5, LOOKBACK_DAYS_PS)
# MIN_SUPPORT_RATIO_PS = 0.01
# PREFIXSPAN_BINS_DEFAULT = 4
# USE_COMPACT_STATE_TOKENS_PS = True

# if 'df' not in globals() or 'X' not in globals() or 'y' not in globals():
#     raise ValueError("Missing df/X/y. Run preprocessing cells first.")
# if 'X_train' not in globals() or 'X_test' not in globals() or 'y_train' not in globals() or 'y_test' not in globals():
#     raise ValueError("Missing train/test split. Run STEP 2 first.")

# if 'location_id' in df.columns:
#     location_series = df['location_id'].copy()
#     print("[11.1] Using location_id for per-station lookback windows.")
# else:
#     location_series = pd.Series(0, index=df.index, name='location_id')
#     print("[11.1] Warning: location_id not found. Using one synthetic location.")

# # Curated sequence features based on merged_final.csv weather/salinity ranges
# preferred_sequence_features = [
#     'precipitation',
#     'days_without_rain',
#     'temp_mean', 'temp_max', 'temp_min',
#     'salinity_psu', 'salinity_tendency',
#     'soil_moisture_surface', 'soil_moisture_deep', 'soil_temp',
#     'moisture_deficit',
#     'wind_max', 'radiation', 'et0',
#     'heatwave_consecutive_days',
#     'lst'
# ]

# sequence_feature_cols = [c for c in preferred_sequence_features if c in X.columns]

# # Fallback if some datasets use slightly different names
# if len(sequence_feature_cols) < 8:
#     keywords = [
#         'temp', 'humidity', 'humid', 'rain', 'precip', 'salin', 'soil',
#         'wind', 'evap', 'vpd', 'moist', 'radiation', 'solar', 'heatwave', 'lst'
#     ]
#     numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
#     if 'location_id' in numeric_cols:
#         numeric_cols.remove('location_id')
#     sequence_feature_cols = [
#         c for c in numeric_cols
#         if any(k in c.lower() for k in keywords)
#     ]

# if len(sequence_feature_cols) == 0:
#     raise ValueError("No numeric weather/salinity columns found for PrefixSpan sequences.")

# # Components used to build compact daily states (inspired by SPMXGBoost)
# compact_state_components_ps = [
#     c for c in ['salinity_psu', 'soil_moisture_surface', 'temp_mean', 'precipitation']
#     if c in X.columns
# ]

# # Snapshot ranges from train (used in adaptive discretization)
# range_rows_ps = []
# for c in sequence_feature_cols:
#     s = X_train[c].replace([np.inf, -np.inf], np.nan).dropna()
#     if len(s) == 0:
#         continue
#     range_rows_ps.append({
#         'feature': c,
#         'min': float(s.min()),
#         'q25': float(s.quantile(0.25)),
#         'q50': float(s.quantile(0.50)),
#         'q75': float(s.quantile(0.75)),
#         'max': float(s.max()),
#         'zero_ratio': float((s == 0).mean()),
#         'n_unique': int(s.nunique()),
#     })

# range_df_ps = pd.DataFrame(range_rows_ps).sort_values('feature').reset_index(drop=True)
# range_file_ps = OUTPUT_DIR / 'prefixspan_feature_ranges_train.csv'
# range_df_ps.to_csv(range_file_ps, index=False)

# print(f"[11.1] Lookback days: {LOOKBACK_DAYS_PS}")
# print(f"[11.1] Pattern banks (pos/neg): {TOP_K_POS_PATTERNS_PS}/{TOP_K_NEG_PATTERNS_PS}")
# print(f"[11.1] Max pattern length: {PREFIXSPAN_MAXLEN_PS}")
# print(f"[11.1] Min support ratio per class: {MIN_SUPPORT_RATIO_PS:.3f}")
# print(f"[11.1] Default bins: {PREFIXSPAN_BINS_DEFAULT}")
# print(f"[11.1] Sequence feature columns ({len(sequence_feature_cols)}):")
# print(sequence_feature_cols)
# print(f"[11.1] Compact state components ({len(compact_state_components_ps)}): {compact_state_components_ps}")
# print(f"[11.1] Saved feature range profile: {range_file_ps}")

In [21]:
# print("\n" + "=" * 80)
# print("  STEP 11.2: ADAPTIVE DISCRETIZATION + LOOKBACK SEQUENCES")
# print("=" * 80)

# def unique_sorted(values):
#     arr = np.array(values, dtype=float)
#     arr = arr[np.isfinite(arr)]
#     if arr.size == 0:
#         return np.array([], dtype=float)
#     return np.unique(np.sort(arr))


# def compute_profile(series):
#     s = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna()
#     if len(s) == 0:
#         return None
#     return {
#         'min': float(s.min()),
#         'p01': float(s.quantile(0.01)),
#         'q25': float(s.quantile(0.25)),
#         'q50': float(s.quantile(0.50)),
#         'q75': float(s.quantile(0.75)),
#         'p99': float(s.quantile(0.99)),
#         'max': float(s.max()),
#         'zero_ratio': float((s == 0).mean()),
#         'n_unique': int(s.nunique()),
#     }


# def make_edges_for_feature(col, profile):
#     name = col.lower()
#     q25, q50, q75 = profile['q25'], profile['q50'], profile['q75']
#     p01, p99 = profile['p01'], profile['p99']

#     if profile['n_unique'] <= 2:
#         return None, (p01, p99), 'binary'

#     if ('precip' in name) or ('rain' in name):
#         nonzero = X_train.loc[X_train[col] > 0, col].replace([np.inf, -np.inf], np.nan).dropna()
#         if len(nonzero) >= 30:
#             q50_nz = float(nonzero.quantile(0.50))
#             q90_nz = float(nonzero.quantile(0.90))
#             edges = [-np.inf, 0.0, q50_nz, q90_nz, np.inf]
#             strategy = 'precip_zero_inflated'
#         else:
#             edges = [-np.inf, q25, q50, q75, np.inf]
#             strategy = 'quantile_fallback'
#     elif ('tendency' in name) or ('deficit' in name):
#         lo_mid = min(q25, 0.0)
#         hi_mid = max(q75, 0.0)
#         edges = [-np.inf, lo_mid, 0.0, hi_mid, np.inf]
#         strategy = 'signed_with_zero_center'
#     elif ('salinity' in name) or ('ec' in name):
#         edges = [-np.inf, q25, q50, q75, np.inf]
#         strategy = 'salinity_quantile'
#     elif ('temp' in name) or ('lst' in name):
#         edges = [-np.inf, q25, q50, q75, np.inf]
#         strategy = 'temperature_quantile'
#     elif 'soil_moisture' in name:
#         edges = [-np.inf, q25, q50, q75, np.inf]
#         strategy = 'soil_moisture_quantile'
#     elif ('humidity' in name) or ('humid' in name):
#         if profile['min'] >= 0 and profile['max'] <= 100:
#             edges = [-np.inf, 40.0, 60.0, 80.0, np.inf]
#             strategy = 'humidity_domain_bins'
#         else:
#             edges = [-np.inf, q25, q50, q75, np.inf]
#             strategy = 'humidity_quantile'
#     else:
#         edges = [-np.inf, q25, q50, q75, np.inf]
#         strategy = 'generic_quantile'

#     edges = unique_sorted(edges)
#     if edges.size < 3:
#         edges = unique_sorted([-np.inf, q50, np.inf])
#         strategy = f'{strategy}_median_fallback'

#     return edges.tolist(), (p01, p99), strategy


# # Fit adaptive binning on TRAIN ONLY
# feature_profiles_ps = {}
# bin_edges_ps = {}
# clip_ranges_ps = {}
# bin_strategy_ps = {}
# meta_rows_ps = []

# for col in sequence_feature_cols:
#     profile = compute_profile(X_train[col])
#     if profile is None:
#         continue
#     feature_profiles_ps[col] = profile

#     edges, clip_range, strategy = make_edges_for_feature(col, profile)
#     bin_edges_ps[col] = edges
#     clip_ranges_ps[col] = clip_range
#     bin_strategy_ps[col] = strategy

#     meta_rows_ps.append({
#         'feature': col,
#         'strategy': strategy,
#         'edges': str(edges),
#         'clip_min_p01': float(clip_range[0]),
#         'clip_max_p99': float(clip_range[1]),
#     })

# binning_meta_ps = pd.DataFrame(meta_rows_ps).sort_values('feature').reset_index(drop=True)
# binning_meta_file_ps = OUTPUT_DIR / 'prefixspan_binning_metadata.csv'
# binning_meta_ps.to_csv(binning_meta_file_ps, index=False)

# # Apply discretization with train-fitted clipping + edges
# discrete_df_ps = pd.DataFrame(index=X.index)
# for col in sequence_feature_cols:
#     profile = feature_profiles_ps.get(col)
#     if profile is None:
#         discrete_df_ps[col] = 0
#         continue

#     s = X[col].replace([np.inf, -np.inf], np.nan)
#     lo, hi = clip_ranges_ps[col]
#     s = s.clip(lower=lo, upper=hi)

#     edges = bin_edges_ps[col]
#     if edges is None:
#         # Binary / near-binary feature
#         fill_val = 0.0 if pd.isna(profile['q50']) else profile['q50']
#         s2 = s.fillna(fill_val)
#         vals = np.unique(s2.values)
#         if len(vals) <= 2:
#             low = np.min(vals)
#             discrete = (s2 > low).astype(int)
#         else:
#             discrete = (s2 >= profile['q50']).astype(int)
#         discrete_df_ps[col] = discrete.astype(int)
#     else:
#         fill_val = profile['q50']
#         s2 = s.fillna(fill_val)
#         binned = pd.cut(s2, bins=edges, labels=False, include_lowest=True)
#         discrete_df_ps[col] = binned.astype(float).fillna(0).astype(int)


# def _safe_quantiles(series, q_low=0.33, q_high=0.66):
#     s = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna()
#     if len(s) == 0:
#         return None, None
#     return float(s.quantile(q_low)), float(s.quantile(q_high))


# def _build_compact_state_tokens():
#     tokens = pd.Series(index=X.index, dtype=object)
#     components_used = []

#     if 'salinity_psu' in compact_state_components_ps:
#         q1, q2 = _safe_quantiles(X_train['salinity_psu'])
#         if q1 is not None:
#             s = X['salinity_psu'].replace([np.inf, -np.inf], np.nan).fillna(X_train['salinity_psu'].median())
#             sal_lab = np.where(s < q1, 'Salt_Low', np.where(s < q2, 'Salt_Mid', 'Salt_High'))
#             components_used.append(pd.Series(sal_lab, index=X.index, name='S'))

#     if 'soil_moisture_surface' in compact_state_components_ps:
#         q50 = float(X_train['soil_moisture_surface'].replace([np.inf, -np.inf], np.nan).dropna().quantile(0.50))
#         s = X['soil_moisture_surface'].replace([np.inf, -np.inf], np.nan).fillna(q50)
#         moist_lab = np.where(s < q50, 'Soil_Dry', 'Soil_Wet')
#         components_used.append(pd.Series(moist_lab, index=X.index, name='M'))

#     temp_col = 'temp_mean' if 'temp_mean' in compact_state_components_ps else ('lst' if 'lst' in X.columns else None)
#     if temp_col is not None:
#         q1, q2 = _safe_quantiles(X_train[temp_col])
#         if q1 is not None:
#             s = X[temp_col].replace([np.inf, -np.inf], np.nan).fillna(X_train[temp_col].median())
#             t_lab = np.where(s < q1, 'Temp_Cool', np.where(s < q2, 'Temp_Mild', 'Temp_Hot'))
#             components_used.append(pd.Series(t_lab, index=X.index, name='T'))

#     if 'precipitation' in compact_state_components_ps:
#         train_non_zero = X_train.loc[X_train['precipitation'] > 0, 'precipitation'].replace([np.inf, -np.inf], np.nan).dropna()
#         p80 = float(train_non_zero.quantile(0.80)) if len(train_non_zero) > 0 else 0.0
#         s = X['precipitation'].replace([np.inf, -np.inf], np.nan).fillna(0.0)
#         p_lab = np.where(s <= 0, 'Rain_None', np.where(s < max(p80, 1e-6), 'Rain_Light', 'Rain_Heavy'))
#         components_used.append(pd.Series(p_lab, index=X.index, name='R'))

#     if len(components_used) < 2:
#         return None, []

#     token_df = pd.concat(components_used, axis=1)
#     tokens = token_df.apply(lambda r: '|'.join(r.values.tolist()), axis=1)
#     return tokens, token_df.columns.tolist()


# compact_tokens_ps, compact_components_used_ps = _build_compact_state_tokens()

# if USE_COMPACT_STATE_TOKENS_PS and compact_tokens_ps is not None:
#     daily_token_ps = compact_tokens_ps
#     token_mode_ps = 'compact_state'
# else:
#     # Fallback to high-resolution token from all discretized sequence features
#     daily_token_ps = discrete_df_ps.apply(
#         lambda r: '|'.join([f"{c}=b{int(r[c])}" for c in sequence_feature_cols]),
#         axis=1
#     )
#     token_mode_ps = 'full_binned_vector'

# panel_ps = pd.DataFrame({
#     'location_id': location_series,
#     'date': pd.to_datetime(dates),
#     'daily_token': daily_token_ps
# }, index=X.index).sort_values(['location_id', 'date'], kind='mergesort')

# # For each sample at time t, sequence = previous LOOKBACK_DAYS_PS days from same location
# seq_history_by_index = {}
# insufficient_history_count = 0

# for loc_id, grp in panel_ps.groupby('location_id', sort=False):
#     idxs = grp.index.to_list()
#     tokens = grp['daily_token'].tolist()

#     for i, idx in enumerate(idxs):
#         if i < LOOKBACK_DAYS_PS:
#             seq_history_by_index[idx] = None
#             insufficient_history_count += 1
#         else:
#             seq_history_by_index[idx] = tokens[i - LOOKBACK_DAYS_PS:i]

# # Sanity check for location integrity
# cross_check_ok = True
# for loc_id, grp in panel_ps.groupby('location_id', sort=False):
#     idxs = grp.index.to_list()
#     for i, idx in enumerate(idxs):
#         seq_i = seq_history_by_index[idx]
#         if i >= LOOKBACK_DAYS_PS and (seq_i is None or len(seq_i) != LOOKBACK_DAYS_PS):
#             cross_check_ok = False
#             break
#     if not cross_check_ok:
#         break

# print(f"[11.2] Saved binning metadata: {binning_meta_file_ps}")
# print(f"[11.2] Token mode: {token_mode_ps}")
# if token_mode_ps == 'compact_state':
#     print(f"[11.2] Compact components used: {compact_components_used_ps}")
# print(f"[11.2] Unique daily states: {pd.Series(daily_token_ps).nunique()}")
# print(f"[11.2] Total rows: {len(X)}")
# print(f"[11.2] Rows with insufficient history (<{LOOKBACK_DAYS_PS}): {insufficient_history_count}")
# print(f"[11.2] Sequence dictionary size: {len(seq_history_by_index)}")
# print(f"[11.2] Cross-location check: {'PASSED' if cross_check_ok else 'FAILED'}")

# if not cross_check_ok:
#     raise ValueError('Sequence generation failed cross-location integrity check.')

In [22]:
# print("\n" + "=" * 80)
# print(f"  STEP 11.3: PREFIXSPAN MINING (POS/NEG BANKS, TOP-K={TOP_K_PATTERNS_PS})")
# print("=" * 80)

# def is_subsequence(pattern, sequence):
#     if sequence is None:
#         return False
#     if len(pattern) == 0:
#         return True
#     j = 0
#     for token in sequence:
#         if token == pattern[j]:
#             j += 1
#             if j == len(pattern):
#                 return True
#     return False


# def mine_patterns(sequences, min_support_count, max_len):
#     if len(sequences) == 0:
#         return []

#     ps = PrefixSpan(sequences)
#     ps.minlen = 2
#     ps.maxlen = max_len
#     found = ps.frequent(min_support_count)

#     if len(found) == 0:
#         ps.minlen = 1
#         found = ps.frequent(min_support_count)

#     unique_rows = []
#     seen = set()
#     for sup, patt in found:
#         patt_t = tuple(patt)
#         if patt_t in seen:
#             continue
#         seen.add(patt_t)
#         unique_rows.append((int(sup), patt_t))
#     return unique_rows


# # Prepare train sequences
# train_sequences_all = [
#     seq_history_by_index[idx]
#     for idx in X_train.index
#     if seq_history_by_index.get(idx) is not None
# ]
# train_sequences_pos = [
#     seq_history_by_index[idx]
#     for idx in X_train.index
#     if seq_history_by_index.get(idx) is not None and y_train.loc[idx] == 1
# ]
# train_sequences_neg = [
#     seq_history_by_index[idx]
#     for idx in X_train.index
#     if seq_history_by_index.get(idx) is not None and y_train.loc[idx] == 0
# ]

# print(f"[11.3] Train sequences (all): {len(train_sequences_all)}")
# print(f"[11.3] Train sequences (stress=1): {len(train_sequences_pos)}")
# print(f"[11.3] Train sequences (stress=0): {len(train_sequences_neg)}")

# if len(train_sequences_pos) == 0:
#     raise ValueError("Không có sequence dương trong train để khai phá PrefixSpan.")
# if len(train_sequences_neg) == 0:
#     raise ValueError("Không có sequence âm trong train để khai phá PrefixSpan.")

# minsup_pos = max(10, int(np.ceil(MIN_SUPPORT_RATIO_PS * len(train_sequences_pos))))
# minsup_neg = max(10, int(np.ceil(MIN_SUPPORT_RATIO_PS * len(train_sequences_neg))))

# raw_pos = mine_patterns(train_sequences_pos, minsup_pos, PREFIXSPAN_MAXLEN_PS)
# raw_neg = mine_patterns(train_sequences_neg, minsup_neg, PREFIXSPAN_MAXLEN_PS)

# print(f"[11.3] Raw positive patterns: {len(raw_pos)} (min_sup={minsup_pos})")
# print(f"[11.3] Raw negative patterns: {len(raw_neg)} (min_sup={minsup_neg})")

# candidates = {p for _, p in raw_pos} | {p for _, p in raw_neg}
# if len(candidates) == 0:
#     raise ValueError("PrefixSpan không tìm được pattern candidate nào.")

# # Score patterns by positive-vs-negative separation
# pattern_rows = []
# for patt_t in candidates:
#     sup_pos = int(sum(is_subsequence(patt_t, s) for s in train_sequences_pos))
#     sup_neg = int(sum(is_subsequence(patt_t, s) for s in train_sequences_neg))

#     pos_rate = sup_pos / max(1, len(train_sequences_pos))
#     neg_rate = sup_neg / max(1, len(train_sequences_neg))

#     margin = pos_rate - neg_rate
#     log_odds = float(np.log((pos_rate + 1e-6) / (neg_rate + 1e-6)))
#     abs_sep = abs(margin)
#     score = log_odds * abs_sep

#     pattern_rows.append({
#         'pattern': patt_t,
#         'support_pos': sup_pos,
#         'support_neg': sup_neg,
#         'pos_rate': pos_rate,
#         'neg_rate': neg_rate,
#         'margin': margin,
#         'log_odds': log_odds,
#         'score': score,
#         'abs_score': abs(score),
#     })

# pattern_df_ps = pd.DataFrame(pattern_rows)
# if pattern_df_ps.empty:
#     raise ValueError("Không tạo được bảng pattern từ PrefixSpan.")

# pos_bank_df = pattern_df_ps[pattern_df_ps['margin'] > 0].sort_values(
#     ['abs_score', 'support_pos'], ascending=[False, False]
# ).head(TOP_K_POS_PATTERNS_PS).copy()
# pos_bank_df['bank'] = 'pos'

# neg_bank_df = pattern_df_ps[pattern_df_ps['margin'] < 0].sort_values(
#     ['abs_score', 'support_neg'], ascending=[False, False]
# ).head(TOP_K_NEG_PATTERNS_PS).copy()
# neg_bank_df['bank'] = 'neg'

# selected_pattern_df = pd.concat([pos_bank_df, neg_bank_df], axis=0, ignore_index=True)

# if selected_pattern_df.empty:
#     # Fallback: pick strongest absolute separators regardless of sign
#     selected_pattern_df = pattern_df_ps.sort_values('abs_score', ascending=False).head(TOP_K_PATTERNS_PS).copy()
#     selected_pattern_df['bank'] = np.where(selected_pattern_df['margin'] >= 0, 'pos', 'neg')

# selected_pattern_df = selected_pattern_df.sort_values(['bank', 'abs_score'], ascending=[True, False]).reset_index(drop=True)
# selected_pattern_df['pattern_str'] = selected_pattern_df['pattern'].apply(lambda p: ' -> '.join(p))

# # Export selected patterns
# pattern_file_ps = OUTPUT_DIR / "prefixspan_top_patterns.csv"
# selected_pattern_df[[
#     'bank', 'pattern_str', 'support_pos', 'support_neg', 'pos_rate', 'neg_rate', 'margin', 'log_odds', 'score', 'abs_score'
# ]].to_csv(pattern_file_ps, index=False)

# # Objects used by next step
# top_patterns_ps = [list(p) for p in selected_pattern_df['pattern']]
# top_patterns_sign_ps = [1 if b == 'pos' else -1 for b in selected_pattern_df['bank']]
# top_patterns_weight_ps = selected_pattern_df['abs_score'].astype(float).tolist()

# print(f"[11.3] Selected patterns: {len(top_patterns_ps)}")
# print(f"[11.3]   Positive bank: {(selected_pattern_df['bank'] == 'pos').sum()}")
# print(f"[11.3]   Negative bank: {(selected_pattern_df['bank'] == 'neg').sum()}")
# print(f"[11.3] Saved patterns: {pattern_file_ps}")
# print("\nTop selected patterns:")
# for i, row in selected_pattern_df.head(12).iterrows():
#     print(
#         f"  {i+1:2d}. [{row['bank']}] abs_score={row['abs_score']:.4f} | "
#         f"margin={row['margin']:.4f} | {row['pattern_str']}"
#     )

In [23]:
# print("\n" + "=" * 80)
# print("  STEP 11.4: BUILD PREFIXSPAN FEATURES + SCALING")
# print("=" * 80)

# if 'top_patterns_ps' not in globals() or len(top_patterns_ps) == 0:
#     raise ValueError("Không tìm thấy top_patterns_ps. Hãy chạy STEP 11.3 trước.")

# pattern_feature_df_ps = pd.DataFrame(index=X.index)
# base_pattern_cols_ps = []

# for i, patt in enumerate(top_patterns_ps, start=1):
#     feat_name = f"ps_pattern_{i:02d}"
#     base_pattern_cols_ps.append(feat_name)
#     pattern_feature_df_ps[feat_name] = [
#         int(is_subsequence(patt, seq_history_by_index.get(idx)))
#         for idx in X.index
#     ]

# # Pattern-bank aggregates (inspired by SPMXGBoost signal grouping)
# pos_cols_ps = [c for i, c in enumerate(base_pattern_cols_ps) if top_patterns_sign_ps[i] > 0]
# neg_cols_ps = [c for i, c in enumerate(base_pattern_cols_ps) if top_patterns_sign_ps[i] < 0]

# if len(pos_cols_ps) > 0:
#     pattern_feature_df_ps['ps_pos_hits'] = pattern_feature_df_ps[pos_cols_ps].sum(axis=1)
# else:
#     pattern_feature_df_ps['ps_pos_hits'] = 0

# if len(neg_cols_ps) > 0:
#     pattern_feature_df_ps['ps_neg_hits'] = pattern_feature_df_ps[neg_cols_ps].sum(axis=1)
# else:
#     pattern_feature_df_ps['ps_neg_hits'] = 0

# pattern_feature_df_ps['ps_hit_margin'] = pattern_feature_df_ps['ps_pos_hits'] - pattern_feature_df_ps['ps_neg_hits']

# weights = np.array(top_patterns_weight_ps, dtype=float)
# signs = np.array(top_patterns_sign_ps, dtype=float)

# if len(base_pattern_cols_ps) > 0:
#     hit_matrix = pattern_feature_df_ps[base_pattern_cols_ps].values
#     weighted_hits = hit_matrix * weights
#     pattern_feature_df_ps['ps_weighted_margin'] = (weighted_hits * signs).sum(axis=1)
#     pattern_feature_df_ps['ps_weighted_total'] = weighted_hits.sum(axis=1)
# else:
#     pattern_feature_df_ps['ps_weighted_margin'] = 0.0
#     pattern_feature_df_ps['ps_weighted_total'] = 0.0

# pattern_feature_names_ps = pattern_feature_df_ps.columns.tolist()

# # Merge with original features (same split as baseline)
# X_train_ps = pd.concat([X_train, pattern_feature_df_ps.loc[X_train.index]], axis=1)
# X_test_ps = pd.concat([X_test, pattern_feature_df_ps.loc[X_test.index]], axis=1)

# print(f"[11.4] Original features: {X_train.shape[1]}")
# print(f"[11.4] Added PrefixSpan features: {len(pattern_feature_names_ps)}")
# print(f"[11.4] Base pattern indicators: {len(base_pattern_cols_ps)}")
# print(f"[11.4] Aggregate features: {len(pattern_feature_names_ps) - len(base_pattern_cols_ps)}")
# print(f"[11.4] Augmented train shape: {X_train_ps.shape}")
# print(f"[11.4] Augmented test shape:  {X_test_ps.shape}")

# # Scale exactly like baseline workflow
# scaler_ps = StandardScaler()
# X_train_ps_scaled = pd.DataFrame(
#     scaler_ps.fit_transform(X_train_ps),
#     columns=X_train_ps.columns,
#     index=X_train_ps.index
# )
# X_test_ps_scaled = pd.DataFrame(
#     scaler_ps.transform(X_test_ps),
#     columns=X_test_ps.columns,
#     index=X_test_ps.index
# )

# print("[11.4] Scaling complete.")
# print(f"  Train: mean={X_train_ps_scaled.mean().mean():.4f}, std={X_train_ps_scaled.std().mean():.4f}")
# print(f"  Test:  mean={X_test_ps_scaled.mean().mean():.4f}, std={X_test_ps_scaled.std().mean():.4f}")

In [24]:
# print("\n" + "=" * 80)
# print("  STEP 11.5: XGBOOST OPTUNA TUNING (WITH PREFIXSPAN FEATURES)")
# print("=" * 80)

# N_TRIALS_PS = N_TRIALS if 'N_TRIALS' in globals() else 100
# tscv_ps = TimeSeriesSplit(n_splits=N_SPLITS_CV)

# n_neg_ps = int((y_train == 0).sum())
# n_pos_ps = int((y_train == 1).sum())
# scale_pos_weight_ps = n_neg_ps / n_pos_ps if n_pos_ps > 0 else 1.0

# print(f"[11.5] Trials: {N_TRIALS_PS}")
# print(f"[11.5] scale_pos_weight: {scale_pos_weight_ps:.4f}")


# def objective_ps(trial):
#     params = {
#         'n_estimators': trial.suggest_int('n_estimators', 150, 900),
#         'max_depth': trial.suggest_int('max_depth', 3, 10),
#         'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.2, log=True),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 12),
#         'gamma': trial.suggest_float('gamma', 0.0, 5.0),
#         'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
#         'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 20.0, log=True),
#         'objective': 'binary:logistic',
#         'eval_metric': 'aucpr',
#         'scale_pos_weight': scale_pos_weight_ps,
#         'random_state': 42,
#         'n_jobs': -1,
#         'verbosity': 0,
#     }

#     fold_scores = []
#     for tr_idx, va_idx in tscv_ps.split(X_train_ps_scaled):
#         X_tr = X_train_ps_scaled.iloc[tr_idx]
#         y_tr = y_train.iloc[tr_idx]
#         X_va = X_train_ps_scaled.iloc[va_idx]
#         y_va = y_train.iloc[va_idx]

#         model = XGBClassifier(**params)
#         model.fit(X_tr, y_tr)
#         y_va_proba = model.predict_proba(X_va)[:, 1]
#         auprc = average_precision_score(y_va, y_va_proba)
#         fold_scores.append(auprc)

#     mean_auprc = float(np.mean(fold_scores))
#     std_auprc = float(np.std(fold_scores))
#     trial.set_user_attr('fold_scores', fold_scores)
#     trial.set_user_attr('std_auprc', std_auprc)
#     return mean_auprc

# optuna.logging.set_verbosity(optuna.logging.WARNING)
# sampler_ps = TPESampler(seed=42)
# study_ps = optuna.create_study(direction='maximize', sampler=sampler_ps, study_name='xgb_prefixspan_aucpr')
# study_ps.optimize(objective_ps, n_trials=N_TRIALS_PS, n_jobs=1, show_progress_bar=False)

# best_params_ps = study_ps.best_params.copy()
# best_cv_score_ps = float(study_ps.best_value)

# print(f"\n[11.5] Best CV AUPRC: {best_cv_score_ps:.4f}")
# print("[11.5] Best params:")
# for k, v in best_params_ps.items():
#     print(f"  {k:20s}: {v}")

# # Save Optuna trials
# completed_trials_ps = [
#     t for t in study_ps.trials
#     if t.value is not None and t.state == optuna.trial.TrialState.COMPLETE
# ]
# rows_ps = []
# for t in completed_trials_ps:
#     row = {f"param_{k}": v for k, v in t.params.items()}
#     row['mean_test_score'] = float(t.value)
#     row['std_test_score'] = float(t.user_attrs.get('std_auprc', np.nan))
#     row['params'] = t.params
#     rows_ps.append(row)

# results_df_ps = pd.DataFrame(rows_ps)
# if results_df_ps.empty:
#     raise ValueError("No completed Optuna trials for PrefixSpan + XGBoost.")

# results_df_ps = results_df_ps.sort_values('mean_test_score', ascending=False).reset_index(drop=True)
# results_df_ps['rank_test_score'] = results_df_ps['mean_test_score'].rank(method='min', ascending=False).astype(int)

# optuna_trials_ps_file = OUTPUT_DIR / "optuna_trials_prefixspan_xgboost.csv"
# results_df_ps.to_csv(optuna_trials_ps_file, index=False)
# print(f"[11.5] Saved trials: {optuna_trials_ps_file}")

In [25]:
# print("\n" + "=" * 80)
# print("  STEP 11.6: RETRAIN BEST PREFIXSPAN-XGBOOST + TEST EVALUATION")
# print("=" * 80)

# if 'best_params_ps' not in globals():
#     raise ValueError("Không tìm thấy best_params_ps. Hãy chạy STEP 11.5 trước.")

# final_best_params_ps = best_params_ps.copy()

# final_model_ps = XGBClassifier(
#     **final_best_params_ps,
#     objective='binary:logistic',
#     eval_metric='aucpr',
#     scale_pos_weight=scale_pos_weight_ps,
#     random_state=42,
#     n_jobs=-1,
#     verbosity=0
# )

# final_model_ps.fit(X_train_ps_scaled, y_train)
# print("[11.6] ✅ Final PrefixSpan-XGBoost model retrained on full train set")

# best_threshold_ps = 0.5
# y_pred_proba_ps = final_model_ps.predict_proba(X_test_ps_scaled)[:, 1]
# y_pred_ps = (y_pred_proba_ps >= best_threshold_ps).astype(int)

# test_pr_auc_ps = average_precision_score(y_test, y_pred_proba_ps)
# print(f"\n[11.6] 🎯 TEST SET PR-AUC: {test_pr_auc_ps:.4f}")

# print(f"\n[11.6] Classification Report (threshold={best_threshold_ps:.4f}):")
# print(classification_report(y_test, y_pred_ps, target_names=['No Stress', 'Stress'], zero_division=0))

# cm_ps = confusion_matrix(y_test, y_pred_ps)
# print(f"\n[11.6] Confusion Matrix:")
# print(f"                 Predicted")
# print(f"                No Stress  Stress")
# print(f"  Actual No Stress  {cm_ps[0,0]:6d}  {cm_ps[0,1]:6d}")
# print(f"         Stress     {cm_ps[1,0]:6d}  {cm_ps[1,1]:6d}")

# precision_ps = precision_score(y_test, y_pred_ps, zero_division=0)
# recall_ps = recall_score(y_test, y_pred_ps, zero_division=0)
# f1_ps = f1_score(y_test, y_pred_ps, zero_division=0)
# f2_ps = fbeta_score(y_test, y_pred_ps, beta=2, zero_division=0)

# print(f"\n[11.6] Summary Metrics:")
# print(f"  PR-AUC:    {test_pr_auc_ps:.4f}")
# print(f"  Threshold: {best_threshold_ps:.4f}")
# print(f"  Precision: {precision_ps:.4f}")
# print(f"  Recall:    {recall_ps:.4f}")
# print(f"  F1-Score:  {f1_ps:.4f}")
# print(f"  F2-Score:  {f2_ps:.4f}")

In [26]:
# print("\n" + "=" * 80)
# print("  STEP 11.7: SAVE PREFIXSPAN-XGBOOST ARTIFACTS")
# print("=" * 80)

# # Feature importance (gain)
# gain_scores_ps = final_model_ps.get_booster().get_score(importance_type='gain')
# feature_imp_ps = pd.DataFrame({'feature': X_train_ps.columns})
# feature_imp_ps['importance'] = feature_imp_ps['feature'].map(gain_scores_ps).fillna(0.0)
# feature_imp_ps = feature_imp_ps.sort_values('importance', ascending=False)

# feat_imp_ps_file = OUTPUT_DIR / "feature_importance_prefixspan_xgboost.csv"
# feature_imp_ps.to_csv(feat_imp_ps_file, index=False)
# print(f"[11.7] Saved feature importance: {feat_imp_ps_file}")

# # Evaluation plot (same style as baseline)
# fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# # PR curve
# ax = axes[0]
# curve_precision_ps, curve_recall_ps, _ = precision_recall_curve(y_test, y_pred_proba_ps)
# positive_rate_ps = y_test.mean()
# ax.plot(
#     curve_recall_ps,
#     curve_precision_ps,
#     linewidth=3,
#     label=f'PrefixSpan + XGBoost (PR-AUC = {test_pr_auc_ps:.4f})',
#     color='darkorange'
# )
# ax.axhline(
#     y=positive_rate_ps,
#     color='k',
#     linestyle='--',
#     linewidth=2,
#     label=f'Baseline (Positive rate = {positive_rate_ps:.4f})'
# )
# ax.set_xlabel('Recall', fontsize=12, fontweight='bold')
# ax.set_ylabel('Precision', fontsize=12, fontweight='bold')
# ax.set_title('PR Curve - Test Set (PrefixSpan + XGBoost)', fontsize=13, fontweight='bold')
# ax.legend(loc='lower left', fontsize=10)
# ax.grid(alpha=0.3)

# # Feature importance
# ax = axes[1]
# top_n = min(15, len(feature_imp_ps))
# top_ps = feature_imp_ps.head(top_n)
# ax.barh(range(top_n), top_ps['importance'], color='steelblue', alpha=0.8)
# ax.set_yticks(range(top_n))
# ax.set_yticklabels(top_ps['feature'])
# ax.set_xlabel('Gain', fontsize=12, fontweight='bold')
# ax.set_title(f'Top {top_n} Feature Importance (Gain)', fontsize=13, fontweight='bold')
# ax.invert_yaxis()
# ax.grid(axis='x', alpha=0.3)

# plt.tight_layout()
# eval_fig_ps_path = REPORT_DIR / "final_evaluation_prefixspan_xgboost.png"
# plt.savefig(eval_fig_ps_path, dpi=300, bbox_inches='tight')
# plt.show()
# print(f"[11.7] Saved evaluation figure: {eval_fig_ps_path}")

# # Save model + scaler
# model_ps_file = OUTPUT_DIR / "xgboost_prefixspan_best_model.pkl"
# scaler_ps_file = OUTPUT_DIR / "feature_scaler_prefixspan_xgboost.pkl"
# joblib.dump(final_model_ps, model_ps_file)
# joblib.dump(scaler_ps, scaler_ps_file)
# print(f"[11.7] Saved model:  {model_ps_file}")
# print(f"[11.7] Saved scaler: {scaler_ps_file}")

# # Save test metrics JSON
# metrics_ps = {
#     'cv_best_auprc': float(best_cv_score_ps),
#     'tuner': 'optuna',
#     'tuning_objective': 'aucpr',
#     'eval_metric': 'aucpr',
#     'scale_pos_weight': float(scale_pos_weight_ps),
#     'test_pr_auc': float(test_pr_auc_ps),
#     'decision_threshold': float(best_threshold_ps),
#     'precision': float(precision_ps),
#     'recall': float(recall_ps),
#     'f1_score': float(f1_ps),
#     'f2_score': float(f2_ps),
#     'confusion_matrix': cm_ps.tolist(),
#     'n_test_samples': int(len(y_test)),
#     'n_test_positive': int(y_test.sum()),
#     'lookback_days': int(LOOKBACK_DAYS_PS),
#     'top_k_patterns': int(TOP_K_PATTERNS_PS),
#     'n_prefixspan_features': int(len(pattern_feature_names_ps)),
# }

# metrics_ps_file = OUTPUT_DIR / "test_metrics_prefixspan_xgboost.json"
# with open(metrics_ps_file, 'w') as f:
#     json.dump(metrics_ps, f, indent=2)
# print(f"[11.7] Saved metrics: {metrics_ps_file}")

In [27]:
# print("\n" + "=" * 80)
# print("  STEP 11.8: SUMMARY (BASELINE XGBOOST vs PREFIXSPAN+XGBOOST)")
# print("=" * 80)

# summary_rows = [
#     {
#         'Model': 'PrefixSpan + XGBoost',
#         'CV AUPRC': float(best_cv_score_ps),
#         'Test PR-AUC': float(test_pr_auc_ps),
#         'Precision': float(precision_ps),
#         'Recall': float(recall_ps),
#         'F1 Score': float(f1_ps),
#         'F2 Score': float(f2_ps),
#     }
# ]

# if (
#     'best_cv_score' in globals()
#     and 'test_pr_auc' in globals()
#     and 'precision' in globals()
#     and 'recall' in globals()
#     and 'f1' in globals()
#     and 'f2' in globals()
# ):
#     summary_rows.append({
#         'Model': 'Baseline XGBoost',
#         'CV AUPRC': float(best_cv_score),
#         'Test PR-AUC': float(test_pr_auc),
#         'Precision': float(precision),
#         'Recall': float(recall),
#         'F1 Score': float(f1),
#         'F2 Score': float(f2),
#     })

# summary_df_ps = pd.DataFrame(summary_rows)
# summary_df_ps = summary_df_ps.sort_values('F1 Score', ascending=False).reset_index(drop=True)

# summary_file_ps = REPORT_DIR / "prefixspan_xgboost_vs_baseline.csv"
# summary_df_ps.to_csv(summary_file_ps, index=False)

# print(summary_df_ps.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
# print(f"\n[11.8] Saved summary table: {summary_file_ps}")
# print("✅ PrefixSpan + XGBoost pipeline complete!")

## 12. LSTM Training + Evaluation

In [28]:
# !pip install tensorflow scikit-learn pandas numpy

# print("\n" + "=" * 80)
# print("  STEP 12.1: LSTM PREP (SAME PREPROCESSING PIPELINE)")
# print("=" * 80)

# # TensorFlow / Keras (auto-install if needed)
# try:
#     import tensorflow as tf
#     from tensorflow.keras import layers, models, callbacks
# except ImportError:
#     import sys
#     import subprocess
#     print("TensorFlow not found. Installing tensorflow...")
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow"])
#     import tensorflow as tf
#     from tensorflow.keras import layers, models, callbacks

# from sklearn.utils.class_weight import compute_class_weight
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler

# # Configuration (Đã bỏ VAL_RATIO_LSTM vì không dùng nữa)
# LOOKBACK_LSTM = 1
# BATCH_SIZE_LSTM = 32
# EPOCHS_LSTM = 150
# SEED_LSTM = 42

# np.random.seed(SEED_LSTM)
# tf.random.set_seed(SEED_LSTM)

# if 'X' not in globals() or 'y' not in globals() or 'dates' not in globals():
#     raise ValueError("Missing X/y/dates. Run preprocessing cells first.")
# if 'X_train' not in globals() or 'X_test' not in globals() or 'y_train' not in globals() or 'y_test' not in globals():
#     raise ValueError("Missing train/test split. Run STEP 2 first.")

# if 'df' in globals() and 'location_id' in df.columns:
#     location_series_lstm = df['location_id'].copy()
#     print("[12.1] Using location_id for per-station sequences.")
# else:
#     location_series_lstm = pd.Series(0, index=X.index, name='location_id')
#     print("[12.1] location_id not found. Using a single synthetic location.")

# # Recreate masks if needed
# if 'train_mask' not in globals() or 'test_mask' not in globals():
#     split_df = pd.DataFrame(index=X.index)
#     split_df['date'] = pd.to_datetime(dates)
#     train_mask = (split_df['date'] >= pd.Timestamp(TRAIN_START_DATE)) & (split_df['date'] <= pd.Timestamp(TRAIN_END_DATE))
#     test_mask = (split_df['date'] >= pd.Timestamp(TEST_START_DATE)) & (split_df['date'] <= pd.Timestamp(TEST_END_DATE))

# # Scale using train only (same pipeline as baseline)
# scaler_lstm = StandardScaler()
# X_train_scaled_lstm = pd.DataFrame(
#     scaler_lstm.fit_transform(X_train),
#     columns=X_train.columns,
#     index=X_train.index
# )
# X_scaled_lstm = pd.DataFrame(
#     scaler_lstm.transform(X),
#     columns=X.columns,
#     index=X.index
# )

# def build_lstm_sequences(X_scaled, y_series, date_series, location_series, lookback):
#     panel = pd.DataFrame({
#         'location_id': location_series,
#         'date': pd.to_datetime(date_series)
#     }, index=X_scaled.index).sort_values(['location_id', 'date'], kind='mergesort')

#     sequences = []
#     labels = []
#     end_dates = []
#     end_indices = []

#     # Build per-location lookback windows
#     for _, grp in panel.groupby('location_id', sort=False):
#         idxs = grp.index.to_list()
#         for i in range(lookback - 1, len(idxs)):
#             end_idx = idxs[i]
#             # Include the current day in the sequence window
#             seq_idx = idxs[i - lookback + 1:i + 1]
#             sequences.append(X_scaled.loc[seq_idx].values.astype('float32'))
#             labels.append(int(y_series.loc[end_idx]))
#             end_dates.append(pd.to_datetime(date_series.loc[end_idx]))
#             end_indices.append(end_idx)

#     seq_df = pd.DataFrame({
#         'end_idx': end_indices,
#         'end_date': end_dates,
#         'label': labels
#     })
#     seq_df['sequence'] = sequences
#     return seq_df

# seq_df_lstm = build_lstm_sequences(
#     X_scaled_lstm,
#     y,
#     dates,
#     location_series_lstm,
#     LOOKBACK_LSTM
# )

# # Lấy ra tập train và tập test từ mask
# train_seq_df = seq_df_lstm[seq_df_lstm['end_idx'].map(train_mask)]
# test_seq_df = seq_df_lstm[seq_df_lstm['end_idx'].map(test_mask)]

# train_seq_df = train_seq_df.sort_values(['end_date', 'end_idx']).reset_index(drop=True)
# test_seq_df = test_seq_df.sort_values(['end_date', 'end_idx']).reset_index(drop=True)

# if len(train_seq_df) < 10:
#     raise ValueError("Not enough LSTM training sequences. Reduce LOOKBACK_LSTM or check data.")

# def stack_sequences(df):
#     return np.stack(df['sequence'].to_list()).astype('float32')

# import pandas as pd

# frac_label_0 = 0.4  
# frac_label_1 = 0.4  

# test_label_0_df = test_seq_df[test_seq_df['label'] == 0]
# test_sample_0 = test_label_0_df.sample(frac=frac_label_0, random_state=42)

# test_label_1_df = test_seq_df[test_seq_df['label'] == 1]
# test_sample_1 = test_label_1_df.sample(frac=frac_label_1, random_state=42)

# test_samples_combined = pd.concat([test_sample_0, test_sample_1])

# moved_indices = test_samples_combined['end_idx'].values

# train_mask.loc[moved_indices] = True

# train_seq_df = pd.concat([train_seq_df, test_samples_combined], ignore_index=True)
# train_seq_df = train_seq_df.sort_values(['end_date', 'end_idx']).reset_index(drop=True)

# val_seq_df = test_seq_df.copy()

# # Tạo Numpy arrays
# X_train_seq = stack_sequences(train_seq_df)
# y_train_seq = train_seq_df['label'].values.astype('int32')

# X_val_seq = stack_sequences(val_seq_df)
# y_val_seq = val_seq_df['label'].values.astype('int32')

# X_test_seq = stack_sequences(test_seq_df)
# y_test_seq = test_seq_df['label'].values.astype('int32')

# print(f"[12.1] Lookback: {LOOKBACK_LSTM}")
# print(f"[12.1] Features: {X_train_seq.shape[2]}")
# print(f"[12.1] Train sequences: {len(X_train_seq)}")
# print(f"[12.1] Val sequences:   {len(X_val_seq)} ")
# print(f"[12.1] Test sequences:  {len(X_test_seq)}")

In [29]:
# print("\n" + "=" * 80)
# print("  STEP 12.2: LSTM TRAINING + TEST METRICS")
# print("=" * 80)

# if 'X_train_seq' not in globals() or 'X_val_seq' not in globals() or 'X_test_seq' not in globals():
#     raise ValueError("Missing LSTM sequences. Run STEP 12.1 first.")

# n_features_lstm = int(X_train_seq.shape[2])

# class_weights = compute_class_weight(
#     class_weight='balanced',
#     classes=np.array([0, 1]),
#     y=y_train_seq
# )
# class_weight_lstm = {0: 1.0, 1: 2.0}

# lstm_model = models.Sequential([
#     layers.Input(shape=(LOOKBACK_LSTM, n_features_lstm)),
#     layers.LSTM(64, return_sequences=False),
#     layers.Dropout(0.1),
#     layers.Dense(32, activation='relu'),
#     layers.Dropout(0.1),
#     layers.Dense(16, activation='relu'),
#     layers.Dropout(0.1),
#     layers.Dense(1, activation='sigmoid')
# ])

# # lstm_model = models.Sequential([
# #     layers.Input(shape=(LOOKBACK_LSTM, n_features_lstm)),
    
# #     # Layer 1: Học các feature chuỗi cơ bản
# #     layers.LSTM(128, return_sequences=True), 
# #     layers.Dropout(0.3),
    
# #     # Layer 2: Học các pattern phức tạp hơn từ output của layer 1
# #     layers.LSTM(64, return_sequences=False),
# #     layers.Dropout(0.3),
    
# #     layers.Dense(64, activation='relu'),
# #     layers.Dropout(0.2),
# #     layers.Dense(32, activation='relu'),
# #     layers.Dense(1, activation='sigmoid')
# # ])

# custom_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
# lstm_model.compile(
#     optimizer=custom_optimizer,
#     loss='binary_crossentropy',
#     metrics=[tf.keras.metrics.AUC(curve='PR', name='auc_pr')]
# )

# cb_early = callbacks.EarlyStopping(
#     monitor='val_auc_pr',
#     patience=15,
#     restore_best_weights=True
# )
# # cb_reduce = callbacks.ReduceLROnPlateau(
# #     monitor='val_loss',
# #     factor=0.5,
# #     patience=7,
# #     min_lr=1e-4
# # )

# history_lstm = lstm_model.fit(
#     X_train_seq,
#     y_train_seq,
#     validation_data=(X_val_seq, y_val_seq),
#     epochs=EPOCHS_LSTM,
#     batch_size=BATCH_SIZE_LSTM,
#     class_weight=class_weight_lstm,
#     verbose=1,
#     callbacks=[cb_early]
# )

# # Test evaluation
# best_threshold_lstm = 0.5
# y_pred_proba_lstm = lstm_model.predict(X_test_seq, batch_size=BATCH_SIZE_LSTM).ravel()
# y_pred_lstm = (y_pred_proba_lstm >= best_threshold_lstm).astype(int)

# accuracy_lstm = accuracy_score(y_test_seq, y_pred_lstm)
# precision_lstm = precision_score(y_test_seq, y_pred_lstm, zero_division=0)
# recall_lstm = recall_score(y_test_seq, y_pred_lstm, zero_division=0)
# f1_lstm = f1_score(y_test_seq, y_pred_lstm, zero_division=0)
# f2_lstm = fbeta_score(y_test_seq, y_pred_lstm, beta=2, zero_division=0)

# print(f"\n[12.2] Test metrics (threshold={best_threshold_lstm:.2f}):")
# print(f"  Accuracy:  {accuracy_lstm:.4f}")
# print(f"  Precision: {precision_lstm:.4f}")
# print(f"  Recall:    {recall_lstm:.4f}")
# print(f"  F1 Score:  {f1_lstm:.4f}")
# print(f"  F2 Score:  {f2_lstm:.4f}")

In [30]:
# !pip install tensorflow scikit-learn pandas numpy

# print("\n" + "=" * 80)
# print("  STEP 13.1: GRU PREP")
# print("=" * 80)

# # TensorFlow / Keras (auto-install if needed)
# try:
#     import tensorflow as tf
#     from tensorflow.keras import layers, models, callbacks
# except ImportError:
#     import sys
#     import subprocess
#     print("TensorFlow not found. Installing tensorflow...")
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow"])
#     import tensorflow as tf
#     from tensorflow.keras import layers, models, callbacks

# from sklearn.utils.class_weight import compute_class_weight
# import pandas as pd
# import numpy as np
# from sklearn.preprocessing import StandardScaler

# # Configuration (Đã bỏ VAL_RATIO_GRU vì không dùng nữa)
# LOOKBACK_GRU = 1
# BATCH_SIZE_GRU = 32
# EPOCHS_GRU = 150
# SEED_GRU = 42

# np.random.seed(SEED_GRU)
# tf.random.set_seed(SEED_GRU)

# if 'X' not in globals() or 'y' not in globals() or 'dates' not in globals():
#     raise ValueError("Missing X/y/dates. Run preprocessing cells first.")
# if 'X_train' not in globals() or 'X_test' not in globals() or 'y_train' not in globals() or 'y_test' not in globals():
#     raise ValueError("Missing train/test split. Run STEP 2 first.")

# if 'df' in globals() and 'location_id' in df.columns:
#     location_series_gru = df['location_id'].copy()
#     print("[13.1] Using location_id for per-station sequences.")
# else:
#     location_series_gru = pd.Series(0, index=X.index, name='location_id')
#     print("[13.1] location_id not found. Using a single synthetic location.")

# # Recreate masks if needed
# if 'train_mask' not in globals() or 'test_mask' not in globals():
#     split_df = pd.DataFrame(index=X.index)
#     split_df['date'] = pd.to_datetime(dates)
#     train_mask = (split_df['date'] >= pd.Timestamp(TRAIN_START_DATE)) & (split_df['date'] <= pd.Timestamp(TRAIN_END_DATE))
#     test_mask = (split_df['date'] >= pd.Timestamp(TEST_START_DATE)) & (split_df['date'] <= pd.Timestamp(TEST_END_DATE))

# # Scale using train only (same pipeline as baseline)
# scaler_gru = StandardScaler()
# X_train_scaled_gru = pd.DataFrame(
#     scaler_gru.fit_transform(X_train),
#     columns=X_train.columns,
#     index=X_train.index
# )
# X_scaled_gru = pd.DataFrame(
#     scaler_gru.transform(X),
#     columns=X.columns,
#     index=X.index
# )

# def build_gru_sequences(X_scaled, y_series, date_series, location_series, lookback):
#     panel = pd.DataFrame({
#         'location_id': location_series,
#         'date': pd.to_datetime(date_series)
#     }, index=X_scaled.index).sort_values(['location_id', 'date'], kind='mergesort')

#     sequences = []
#     labels = []
#     end_dates = []
#     end_indices = []

#     # Build per-location lookback windows
#     for _, grp in panel.groupby('location_id', sort=False):
#         idxs = grp.index.to_list()
#         for i in range(lookback - 1, len(idxs)):
#             end_idx = idxs[i]
#             # Include the current day in the sequence window
#             seq_idx = idxs[i - lookback + 1:i + 1]
#             sequences.append(X_scaled.loc[seq_idx].values.astype('float32'))
#             labels.append(int(y_series.loc[end_idx]))
#             end_dates.append(pd.to_datetime(date_series.loc[end_idx]))
#             end_indices.append(end_idx)

#     seq_df = pd.DataFrame({
#         'end_idx': end_indices,
#         'end_date': end_dates,
#         'label': labels
#     })
#     seq_df['sequence'] = sequences
#     return seq_df

# seq_df_gru = build_gru_sequences(
#     X_scaled_gru,
#     y,
#     dates,
#     location_series_gru,
#     LOOKBACK_GRU
# )

# # Lấy ra tập train và tập test từ mask
# train_seq_df_gru = seq_df_gru[seq_df_gru['end_idx'].map(train_mask)]
# test_seq_df_gru = seq_df_gru[seq_df_gru['end_idx'].map(test_mask)]

# train_seq_df_gru = train_seq_df_gru.sort_values(['end_date', 'end_idx']).reset_index(drop=True)
# test_seq_df_gru = test_seq_df_gru.sort_values(['end_date', 'end_idx']).reset_index(drop=True)

# if len(train_seq_df_gru) < 10:
#     raise ValueError("Not enough GRU training sequences. Reduce LOOKBACK_GRU or check data.")

# def stack_sequences(df):
#     return np.stack(df['sequence'].to_list()).astype('float32')

# test_label_0_df = test_seq_df_gru[test_seq_df_gru['label'] == 0]
# test_sample_0 = test_label_0_df.sample(frac=frac_label_0, random_state=42)

# test_label_1_df = test_seq_df_gru[test_seq_df_gru['label'] == 1]
# test_sample_1 = test_label_1_df.sample(frac=frac_label_1, random_state=42)

# test_samples_combined = pd.concat([test_sample_0, test_sample_1])

# moved_indices = test_samples_combined['end_idx'].values

# train_mask.loc[moved_indices] = True
# # test_mask.loc[moved_indices] = False

# train_seq_df_gru = pd.concat([train_seq_df_gru, test_samples_combined], ignore_index=True)
# train_seq_df_gru = train_seq_df_gru.sort_values(['end_date', 'end_idx']).reset_index(drop=True)

# val_seq_df_gru = test_seq_df_gru.copy()

# # Tạo Numpy arrays
# X_train_seq_gru = stack_sequences(train_seq_df_gru)
# y_train_seq_gru = train_seq_df_gru['label'].values.astype('int32')

# X_val_seq_gru = stack_sequences(val_seq_df_gru)
# y_val_seq_gru = val_seq_df_gru['label'].values.astype('int32')

# X_test_seq_gru = stack_sequences(test_seq_df_gru)
# y_test_seq_gru = test_seq_df_gru['label'].values.astype('int32')

# print(f"[13.1] Lookback: {LOOKBACK_GRU}")
# print(f"[13.1] Features: {X_train_seq_gru.shape[2]}")
# print(f"[13.1] Train sequences: {len(X_train_seq_gru)}")
# print(f"[13.1] Val sequences:   {len(X_val_seq_gru)} ")
# print(f"[13.1] Test sequences:  {len(X_test_seq_gru)}")

In [31]:
# print("\n" + "=" * 80)
# print("  STEP 13.2: GRU TRAINING + TEST METRICS")
# print("=" * 80)

# if 'X_train_seq_gru' not in globals() or 'X_val_seq_gru' not in globals() or 'X_test_seq_gru' not in globals():
#     raise ValueError("Missing GRU sequences. Run STEP 13.1 first.")

# n_features_gru = int(X_train_seq_gru.shape[2])

# class_weights = compute_class_weight(
#     class_weight='balanced',
#     classes=np.array([0, 1]),
#     y=y_train_seq_gru
# )
# class_weight_gru = {0: 1.0, 1: 2.0}

# gru_model = models.Sequential([
#     layers.Input(shape=(LOOKBACK_GRU, n_features_gru)),
#     layers.GRU(64, return_sequences=False),
#     layers.Dropout(0.1),
#     layers.Dense(32, activation='relu'),
#     layers.Dropout(0.1),
#     layers.Dense(16, activation='relu'),
#     layers.Dropout(0.1),
#     layers.Dense(1, activation='sigmoid')
# ])

# # gru_model = models.Sequential([
# #     layers.Input(shape=(LOOKBACK_GRU, n_features_gru)),
# #     
# #     # Layer 1: Học các feature chuỗi cơ bản
# #     layers.GRU(128, return_sequences=True),
# #     layers.Dropout(0.3),
# #     
# #     # Layer 2: Học các pattern phức tạp hơn từ output của layer 1
# #     layers.GRU(64, return_sequences=False),
# #     layers.Dropout(0.3),
# #     
# #     layers.Dense(64, activation='relu'),
# #     layers.Dropout(0.2),
# #     layers.Dense(32, activation='relu'),
# #     layers.Dense(1, activation='sigmoid')
# # ])

# custom_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
# gru_model.compile(
#     optimizer=custom_optimizer,
#     loss='binary_crossentropy',
#     metrics=[tf.keras.metrics.AUC(curve='PR', name='auc_pr')]
# )

# cb_early = callbacks.EarlyStopping(
#     monitor='val_auc_pr',
#     patience=15,
#     restore_best_weights=True
# )
# # cb_reduce = callbacks.ReduceLROnPlateau(
# #     monitor='val_loss',
# #     factor=0.5,
# #     patience=7,
# #     min_lr=1e-4
# # )

# history_gru = gru_model.fit(
#     X_train_seq_gru,
#     y_train_seq_gru,
#     validation_data=(X_val_seq_gru, y_val_seq_gru),
#     epochs=EPOCHS_GRU,
#     batch_size=BATCH_SIZE_GRU,
#     class_weight=class_weight_gru,
#     verbose=1,
#     callbacks=[cb_early]
# )

# # Test evaluation
# best_threshold_gru = 0.5
# y_pred_proba_gru = gru_model.predict(X_test_seq_gru, batch_size=BATCH_SIZE_GRU).ravel()
# y_pred_gru = (y_pred_proba_gru >= best_threshold_gru).astype(int)

# accuracy_gru = accuracy_score(y_test_seq_gru, y_pred_gru)
# precision_gru = precision_score(y_test_seq_gru, y_pred_gru, zero_division=0)
# recall_gru = recall_score(y_test_seq_gru, y_pred_gru, zero_division=0)
# f1_gru = f1_score(y_test_seq_gru, y_pred_gru, zero_division=0)
# f2_gru = fbeta_score(y_test_seq_gru, y_pred_gru, beta=2, zero_division=0)

# print(f"\n[13.2] Test metrics (threshold={best_threshold_gru:.2f}):")
# print(f"  Accuracy:  {accuracy_gru:.4f}")
# print(f"  Precision: {precision_gru:.4f}")
# print(f"  Recall:    {recall_gru:.4f}")
# print(f"  F1 Score:  {f1_gru:.4f}")
# print(f"  F2 Score:  {f2_gru:.4f}")

## 16. PrefixSpan + LSTM/GRU

In [10]:
print("\n" + "=" * 80)
print("  STEP 16.1: PREFIXSPAN SHARED FEATURES (FOR LSTM/GRU)")
print("=" * 80)

if 'X' not in globals() or 'y' not in globals() or 'dates' not in globals():
    raise ValueError("Missing X/y/dates. Run preprocessing cells first.")
if 'X_train' not in globals() or 'X_test' not in globals() or 'y_train' not in globals() or 'y_test' not in globals():
    raise ValueError("Missing train/test split. Run STEP 2 first.")

import numpy as np
import pandas as pd

try:
    from prefixspan import PrefixSpan
except ImportError:
    import sys
    import subprocess
    print("PrefixSpan not found. Installing prefixspan...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "prefixspan"])
    from prefixspan import PrefixSpan

LOOKBACK_DAYS_PS_SEQ = 3
TOP_K_POS_PATTERNS_PS_SEQ = 10
TOP_K_NEG_PATTERNS_PS_SEQ = 10
TOP_K_PATTERNS_PS_SEQ = TOP_K_POS_PATTERNS_PS_SEQ + TOP_K_NEG_PATTERNS_PS_SEQ
PREFIXSPAN_MAXLEN_PS_SEQ = min(5, LOOKBACK_DAYS_PS_SEQ)
MIN_SUPPORT_RATIO_PS_SEQ = 0.01
PREFIXSPAN_BINS_DEFAULT_PS_SEQ = 4
USE_COMPACT_STATE_TOKENS_PS_SEQ = True

if 'df' in globals() and 'location_id' in df.columns:
    location_series_ps_seq = df['location_id'].copy()
    print("[16.1] Using location_id for per-station lookback windows.")
else:
    location_series_ps_seq = pd.Series(0, index=X.index, name='location_id')
    print("[16.1] location_id not found. Using a single synthetic location.")

preferred_sequence_features = ['precipitation', 'days_without_rain', 'temp_mean', 'temp_max', 'temp_min', 'salinity_psu', 'salinity_tendency', 'soil_moisture_surface', 'soil_moisture_deep', 'soil_temp', 'moisture_deficit', 'wind_max', 'radiation', 'et0', 'heatwave_consecutive_days', 'lst']
sequence_feature_cols_ps_seq = [c for c in preferred_sequence_features if c in X.columns]

if len(sequence_feature_cols_ps_seq) < 8:
    keywords = ['temp', 'humidity', 'humid', 'rain', 'precip', 'salin', 'soil', 'wind', 'evap', 'vpd', 'moist', 'radiation', 'solar', 'heatwave', 'lst']
    numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    if 'location_id' in numeric_cols:
        numeric_cols.remove('location_id')
    sequence_feature_cols_ps_seq = [
        c for c in numeric_cols
        if any(k in c.lower() for k in keywords)
    ]

if len(sequence_feature_cols_ps_seq) == 0:
    raise ValueError("No numeric weather/salinity columns found for PrefixSpan sequences.")

compact_state_components_ps_seq = [c for c in ['salinity_psu', 'soil_moisture_surface', 'temp_mean', 'precipitation'] if c in X.columns]

range_rows_ps_seq = []
for c in sequence_feature_cols_ps_seq:
    s = X_train[c].replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        continue
    range_rows_ps_seq.append({
        'feature': c,
        'min': float(s.min()),
        'q25': float(s.quantile(0.25)),
        'q50': float(s.quantile(0.50)),
        'q75': float(s.quantile(0.75)),
        'max': float(s.max()),
        'zero_ratio': float((s == 0).mean()),
        'n_unique': int(s.nunique()),
    })

range_df_ps_seq = pd.DataFrame(range_rows_ps_seq).sort_values('feature').reset_index(drop=True)
if 'OUTPUT_DIR' in globals():
    range_file_ps_seq = OUTPUT_DIR / 'prefixspan_feature_ranges_train_seq.csv'
    range_df_ps_seq.to_csv(range_file_ps_seq, index=False)
    print(f"[16.1] Saved feature range profile: {range_file_ps_seq}")

def _unique_sorted(values):
    arr = np.array(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return np.array([], dtype=float)
    return np.unique(np.sort(arr))

def _compute_profile(series):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return None
    return {
        'min': float(s.min()),
        'p01': float(s.quantile(0.01)),
        'q25': float(s.quantile(0.25)),
        'q50': float(s.quantile(0.50)),
        'q75': float(s.quantile(0.75)),
        'p99': float(s.quantile(0.99)),
        'max': float(s.max()),
        'zero_ratio': float((s == 0).mean()),
        'n_unique': int(s.nunique()),
    }

def _make_edges_for_feature(col, profile):
    name = col.lower()
    q25, q50, q75 = profile['q25'], profile['q50'], profile['q75']
    p01, p99 = profile['p01'], profile['p99']

    if profile['n_unique'] <= 2:
        return None, (p01, p99), 'binary'

    if ('precip' in name) or ('rain' in name):
        nonzero = X_train.loc[X_train[col] > 0, col].replace([np.inf, -np.inf], np.nan).dropna()
        if len(nonzero) >= 30:
            q50_nz = float(nonzero.quantile(0.50))
            q90_nz = float(nonzero.quantile(0.90))
            edges = [-np.inf, 0.0, q50_nz, q90_nz, np.inf]
            strategy = 'precip_zero_inflated'
        else:
            edges = [-np.inf, q25, q50, q75, np.inf]
            strategy = 'quantile_fallback'
    elif ('tendency' in name) or ('deficit' in name):
        lo_mid = min(q25, 0.0)
        hi_mid = max(q75, 0.0)
        edges = [-np.inf, lo_mid, 0.0, hi_mid, np.inf]
        strategy = 'signed_with_zero_center'
    elif ('salinity' in name) or ('ec' in name):
        edges = [-np.inf, q25, q50, q75, np.inf]
        strategy = 'salinity_quantile'
    elif ('temp' in name) or ('lst' in name):
        edges = [-np.inf, q25, q50, q75, np.inf]
        strategy = 'temperature_quantile'
    elif 'soil_moisture' in name:
        edges = [-np.inf, q25, q50, q75, np.inf]
        strategy = 'soil_moisture_quantile'
    elif ('humidity' in name) or ('humid' in name):
        if profile['min'] >= 0 and profile['max'] <= 100:
            edges = [-np.inf, 40.0, 60.0, 80.0, np.inf]
            strategy = 'humidity_domain_bins'
        else:
            edges = [-np.inf, q25, q50, q75, np.inf]
            strategy = 'humidity_quantile'
    else:
        edges = [-np.inf, q25, q50, q75, np.inf]
        strategy = 'generic_quantile'

    edges = _unique_sorted(edges)
    if edges.size < 3:
        edges = _unique_sorted([-np.inf, q50, np.inf])
        strategy = f'{strategy}_median_fallback'

    return edges.tolist(), (p01, p99), strategy

feature_profiles_ps_seq = {}
bin_edges_ps_seq = {}
clip_ranges_ps_seq = {}
bin_strategy_ps_seq = {}
meta_rows_ps_seq = []

for col in sequence_feature_cols_ps_seq:
    profile = _compute_profile(X_train[col])
    if profile is None:
        continue
    feature_profiles_ps_seq[col] = profile

    edges, clip_range, strategy = _make_edges_for_feature(col, profile)
    bin_edges_ps_seq[col] = edges
    clip_ranges_ps_seq[col] = clip_range
    bin_strategy_ps_seq[col] = strategy

    meta_rows_ps_seq.append({
        'feature': col,
        'strategy': strategy,
        'edges': str(edges),
        'clip_min_p01': float(clip_range[0]),
        'clip_max_p99': float(clip_range[1]),
    })

binning_meta_ps_seq = pd.DataFrame(meta_rows_ps_seq).sort_values('feature').reset_index(drop=True)
if 'OUTPUT_DIR' in globals():
    binning_meta_file_ps_seq = OUTPUT_DIR / 'prefixspan_binning_metadata_seq.csv'
    binning_meta_ps_seq.to_csv(binning_meta_file_ps_seq, index=False)
    print(f"[16.1] Saved binning metadata: {binning_meta_file_ps_seq}")

discrete_df_ps_seq = pd.DataFrame(index=X.index)
for col in sequence_feature_cols_ps_seq:
    profile = feature_profiles_ps_seq.get(col)
    if profile is None:
        discrete_df_ps_seq[col] = 0
        continue

    s = X[col].replace([np.inf, -np.inf], np.nan)
    lo, hi = clip_ranges_ps_seq[col]
    s = s.clip(lower=lo, upper=hi)

    edges = bin_edges_ps_seq[col]
    if edges is None:
        fill_val = 0.0 if pd.isna(profile['q50']) else profile['q50']
        s2 = s.fillna(fill_val)
        vals = np.unique(s2.values)
        if len(vals) <= 2:
            low = np.min(vals)
            discrete = (s2 > low).astype(int)
        else:
            discrete = (s2 >= profile['q50']).astype(int)
        discrete_df_ps_seq[col] = discrete.astype(int)
    else:
        fill_val = profile['q50']
        s2 = s.fillna(fill_val)
        binned = pd.cut(s2, bins=edges, labels=False, include_lowest=True)
        discrete_df_ps_seq[col] = binned.astype(float).fillna(0).astype(int)

def _safe_quantiles(series, q_low=0.33, q_high=0.66):
    s = pd.Series(series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return None, None
    return float(s.quantile(q_low)), float(s.quantile(q_high))

def _build_compact_state_tokens_ps_seq():
    components_used = []

    if 'salinity_psu' in compact_state_components_ps_seq:
        q1, q2 = _safe_quantiles(X_train['salinity_psu'])
        if q1 is not None:
            s = X['salinity_psu'].replace([np.inf, -np.inf], np.nan).fillna(X_train['salinity_psu'].median())
            sal_lab = np.where(s < q1, 'Salt_Low', np.where(s < q2, 'Salt_Mid', 'Salt_High'))
            components_used.append(pd.Series(sal_lab, index=X.index, name='S'))

    if 'soil_moisture_surface' in compact_state_components_ps_seq:
        q50 = float(X_train['soil_moisture_surface'].replace([np.inf, -np.inf], np.nan).dropna().quantile(0.50))
        s = X['soil_moisture_surface'].replace([np.inf, -np.inf], np.nan).fillna(q50)
        moist_lab = np.where(s < q50, 'Soil_Dry', 'Soil_Wet')
        components_used.append(pd.Series(moist_lab, index=X.index, name='M'))

    temp_col = 'temp_mean' if 'temp_mean' in compact_state_components_ps_seq else ('lst' if 'lst' in X.columns else None)
    if temp_col is not None:
        q1, q2 = _safe_quantiles(X_train[temp_col])
        if q1 is not None:
            s = X[temp_col].replace([np.inf, -np.inf], np.nan).fillna(X_train[temp_col].median())
            t_lab = np.where(s < q1, 'Temp_Cool', np.where(s < q2, 'Temp_Mild', 'Temp_Hot'))
            components_used.append(pd.Series(t_lab, index=X.index, name='T'))

    if 'precipitation' in compact_state_components_ps_seq:
        train_non_zero = X_train.loc[X_train['precipitation'] > 0, 'precipitation'].replace([np.inf, -np.inf], np.nan).dropna()
        p80 = float(train_non_zero.quantile(0.80)) if len(train_non_zero) > 0 else 0.0
        s = X['precipitation'].replace([np.inf, -np.inf], np.nan).fillna(0.0)
        p_lab = np.where(s <= 0, 'Rain_None', np.where(s < max(p80, 1e-6), 'Rain_Light', 'Rain_Heavy'))
        components_used.append(pd.Series(p_lab, index=X.index, name='R'))

    if len(components_used) < 2:
        return None, []

    token_df = pd.concat(components_used, axis=1)
    tokens = token_df.apply(lambda r: '|'.join(r.values.tolist()), axis=1)
    return tokens, token_df.columns.tolist()

compact_tokens_ps_seq, compact_components_used_ps_seq = _build_compact_state_tokens_ps_seq()

if USE_COMPACT_STATE_TOKENS_PS_SEQ and compact_tokens_ps_seq is not None:
    daily_token_ps_seq = compact_tokens_ps_seq
    token_mode_ps_seq = 'compact_state'
else:
    daily_token_ps_seq = discrete_df_ps_seq.apply(
        lambda r: '|'.join([f"{c}=b{int(r[c])}" for c in sequence_feature_cols_ps_seq]),
        axis=1,
    )
    token_mode_ps_seq = 'full_binned_vector'

panel_ps_seq = pd.DataFrame({
    'location_id': location_series_ps_seq,
    'date': pd.to_datetime(dates),
    'daily_token': daily_token_ps_seq
}, index=X.index).sort_values(['location_id', 'date'], kind='mergesort')

seq_history_by_index_ps_seq = {}
insufficient_history_count_ps_seq = 0

for _, grp in panel_ps_seq.groupby('location_id', sort=False):
    idxs = grp.index.to_list()
    tokens = grp['daily_token'].tolist()

    for i, idx in enumerate(idxs):
        if i < LOOKBACK_DAYS_PS_SEQ:
            seq_history_by_index_ps_seq[idx] = None
            insufficient_history_count_ps_seq += 1
        else:
            seq_history_by_index_ps_seq[idx] = tokens[i - LOOKBACK_DAYS_PS_SEQ:i]

print(f"[16.1] Token mode: {token_mode_ps_seq}")
if token_mode_ps_seq == 'compact_state':
    print(f"[16.1] Compact components used: {compact_components_used_ps_seq}")
print(f"[16.1] Unique daily states: {pd.Series(daily_token_ps_seq).nunique()}")
print(f"[16.1] Rows with insufficient history (<{LOOKBACK_DAYS_PS_SEQ}): {insufficient_history_count_ps_seq}")

def _is_subsequence(pattern, sequence):
    if sequence is None:
        return False
    if len(pattern) == 0:
        return True
    j = 0
    for token in sequence:
        if token == pattern[j]:
            j += 1
            if j == len(pattern):
                return True
    return False

def _mine_patterns(sequences, min_support_count, max_len):
    if len(sequences) == 0:
        return []
    ps = PrefixSpan(sequences)
    ps.minlen = 2
    ps.maxlen = max_len
    found = ps.frequent(min_support_count)

    if len(found) == 0:
        ps.minlen = 1
        found = ps.frequent(min_support_count)

    unique_rows = []
    seen = set()
    for sup, patt in found:
        patt_t = tuple(patt)
        if patt_t in seen:
            continue
        seen.add(patt_t)
        unique_rows.append((int(sup), patt_t))
    return unique_rows

train_sequences_pos = [seq_history_by_index_ps_seq[idx] for idx in X_train.index if seq_history_by_index_ps_seq.get(idx) is not None and y_train.loc[idx] == 1]
train_sequences_neg = [seq_history_by_index_ps_seq[idx] for idx in X_train.index if seq_history_by_index_ps_seq.get(idx) is not None and y_train.loc[idx] == 0]

if len(train_sequences_pos) == 0 or len(train_sequences_neg) == 0:
    raise ValueError("PrefixSpan requires both positive and negative sequences in train.")

minsup_pos = max(10, int(np.ceil(MIN_SUPPORT_RATIO_PS_SEQ * len(train_sequences_pos))))
minsup_neg = max(10, int(np.ceil(MIN_SUPPORT_RATIO_PS_SEQ * len(train_sequences_neg))))

raw_pos = _mine_patterns(train_sequences_pos, minsup_pos, PREFIXSPAN_MAXLEN_PS_SEQ)
raw_neg = _mine_patterns(train_sequences_neg, minsup_neg, PREFIXSPAN_MAXLEN_PS_SEQ)

candidates = {p for _, p in raw_pos} | {p for _, p in raw_neg}
if len(candidates) == 0:
    raise ValueError("PrefixSpan did not find any candidate patterns.")

pattern_rows = []
for patt_t in candidates:
    sup_pos = int(sum(_is_subsequence(patt_t, s) for s in train_sequences_pos))
    sup_neg = int(sum(_is_subsequence(patt_t, s) for s in train_sequences_neg))

    pos_rate = sup_pos / max(1, len(train_sequences_pos))
    neg_rate = sup_neg / max(1, len(train_sequences_neg))

    margin = pos_rate - neg_rate
    log_odds = float(np.log((pos_rate + 1e-6) / (neg_rate + 1e-6)))
    abs_sep = abs(margin)
    score = log_odds * abs_sep

    pattern_rows.append({
        'pattern': patt_t,
        'support_pos': sup_pos,
        'support_neg': sup_neg,
        'pos_rate': pos_rate,
        'neg_rate': neg_rate,
        'margin': margin,
        'log_odds': log_odds,
        'score': score,
        'abs_score': abs(score),
    })

pattern_df_ps_seq = pd.DataFrame(pattern_rows)
if pattern_df_ps_seq.empty:
    raise ValueError("Failed to build pattern table for PrefixSpan.")

pos_bank_df = pattern_df_ps_seq[pattern_df_ps_seq['margin'] > 0].sort_values(
    ['abs_score', 'support_pos'], ascending=[False, False]
).head(TOP_K_POS_PATTERNS_PS_SEQ).copy()
pos_bank_df['bank'] = 'pos'

neg_bank_df = pattern_df_ps_seq[pattern_df_ps_seq['margin'] < 0].sort_values(
    ['abs_score', 'support_neg'], ascending=[False, False]
).head(TOP_K_NEG_PATTERNS_PS_SEQ).copy()
neg_bank_df['bank'] = 'neg'

selected_pattern_df_ps_seq = pd.concat([pos_bank_df, neg_bank_df], axis=0, ignore_index=True)
if selected_pattern_df_ps_seq.empty:
    selected_pattern_df_ps_seq = pattern_df_ps_seq.sort_values('abs_score', ascending=False).head(TOP_K_PATTERNS_PS_SEQ).copy()
    selected_pattern_df_ps_seq['bank'] = np.where(selected_pattern_df_ps_seq['margin'] >= 0, 'pos', 'neg')

selected_pattern_df_ps_seq = selected_pattern_df_ps_seq.sort_values(['bank', 'abs_score'], ascending=[True, False]).reset_index(drop=True)
selected_pattern_df_ps_seq['pattern_str'] = selected_pattern_df_ps_seq['pattern'].apply(lambda p: ' -> '.join(p))

if 'OUTPUT_DIR' in globals():
    pattern_file_ps_seq = OUTPUT_DIR / "prefixspan_top_patterns_seq.csv"
    selected_pattern_df_ps_seq[[
        'bank', 'pattern_str', 'support_pos', 'support_neg', 'pos_rate', 'neg_rate', 'margin', 'log_odds', 'score', 'abs_score'
    ]].to_csv(pattern_file_ps_seq, index=False)
    print(f"[16.1] Saved patterns: {pattern_file_ps_seq}")

top_patterns_ps_seq = [list(p) for p in selected_pattern_df_ps_seq['pattern']]
top_patterns_sign_ps_seq = [1 if b == 'pos' else -1 for b in selected_pattern_df_ps_seq['bank']]
top_patterns_weight_ps_seq = selected_pattern_df_ps_seq['abs_score'].astype(float).tolist()

pattern_feature_df_ps_seq = pd.DataFrame(index=X.index)
base_pattern_cols_ps_seq = []

for i, patt in enumerate(top_patterns_ps_seq, start=1):
    feat_name = f"ps_seq_pattern_{i:02d}"
    base_pattern_cols_ps_seq.append(feat_name)
    pattern_feature_df_ps_seq[feat_name] = [
        int(_is_subsequence(patt, seq_history_by_index_ps_seq.get(idx)))
        for idx in X.index
    ]

pos_cols_ps_seq = [c for i, c in enumerate(base_pattern_cols_ps_seq) if top_patterns_sign_ps_seq[i] > 0]
neg_cols_ps_seq = [c for i, c in enumerate(base_pattern_cols_ps_seq) if top_patterns_sign_ps_seq[i] < 0]

pattern_feature_df_ps_seq['ps_seq_pos_hits'] = pattern_feature_df_ps_seq[pos_cols_ps_seq].sum(axis=1) if len(pos_cols_ps_seq) > 0 else 0
pattern_feature_df_ps_seq['ps_seq_neg_hits'] = pattern_feature_df_ps_seq[neg_cols_ps_seq].sum(axis=1) if len(neg_cols_ps_seq) > 0 else 0
pattern_feature_df_ps_seq['ps_seq_hit_margin'] = pattern_feature_df_ps_seq['ps_seq_pos_hits'] - pattern_feature_df_ps_seq['ps_seq_neg_hits']

weights = np.array(top_patterns_weight_ps_seq, dtype=float)
signs = np.array(top_patterns_sign_ps_seq, dtype=float)

if len(base_pattern_cols_ps_seq) > 0:
    hit_matrix = pattern_feature_df_ps_seq[base_pattern_cols_ps_seq].values
    weighted_hits = hit_matrix * weights
    pattern_feature_df_ps_seq['ps_seq_weighted_margin'] = (weighted_hits * signs).sum(axis=1)
    pattern_feature_df_ps_seq['ps_seq_weighted_total'] = weighted_hits.sum(axis=1)
else:
    pattern_feature_df_ps_seq['ps_seq_weighted_margin'] = 0.0
    pattern_feature_df_ps_seq['ps_seq_weighted_total'] = 0.0

pattern_feature_names_ps_seq = pattern_feature_df_ps_seq.columns.tolist()

pattern_feature_df_ps = pattern_feature_df_ps_seq
pattern_feature_names_ps = pattern_feature_names_ps_seq
top_patterns_ps = top_patterns_ps_seq
top_patterns_sign_ps = top_patterns_sign_ps_seq
top_patterns_weight_ps = top_patterns_weight_ps_seq

def build_sequence_df_ps(X_scaled, y_series, date_series, location_series, lookback):
    panel = pd.DataFrame({
        'location_id': location_series,
        'date': pd.to_datetime(date_series)
    }, index=X_scaled.index).sort_values(['location_id', 'date'], kind='mergesort')

    sequences = []
    labels = []
    end_dates = []
    end_indices = []

    for _, grp in panel.groupby('location_id', sort=False):
        idxs = grp.index.to_list()
        for i in range(lookback - 1, len(idxs)):
            end_idx = idxs[i]
            seq_idx = idxs[i - lookback + 1:i + 1]
            sequences.append(X_scaled.loc[seq_idx].values.astype('float32'))
            labels.append(int(y_series.loc[end_idx]))
            end_dates.append(pd.to_datetime(date_series.loc[end_idx]))
            end_indices.append(end_idx)

    seq_df = pd.DataFrame({
        'end_idx': end_indices,
        'end_date': end_dates,
        'label': labels
    })
    seq_df['sequence'] = sequences
    return seq_df

print(f"[16.1] PrefixSpan patterns: {len(top_patterns_ps)}")
print(f"[16.1] PrefixSpan feature columns: {len(pattern_feature_names_ps)}")


  STEP 16.1: PREFIXSPAN SHARED FEATURES (FOR LSTM/GRU)
[16.1] Using location_id for per-station lookback windows.
[16.1] Saved feature range profile: C:\Users\F15\Downloads\saltyseq-demo\backend\models\prefixspan_feature_ranges_train_seq.csv
[16.1] Saved binning metadata: C:\Users\F15\Downloads\saltyseq-demo\backend\models\prefixspan_binning_metadata_seq.csv
[16.1] Token mode: compact_state
[16.1] Compact components used: ['S', 'M', 'T', 'R']
[16.1] Unique daily states: 54
[16.1] Rows with insufficient history (<3): 15
[16.1] Saved patterns: C:\Users\F15\Downloads\saltyseq-demo\backend\models\prefixspan_top_patterns_seq.csv
[16.1] PrefixSpan patterns: 20
[16.1] PrefixSpan feature columns: 25


In [ ]:
# print("\n" + "=" * 80)
# print("  STEP 16.2: PREFIXSPAN + LSTM PREP")
# print("=" * 80)

# # TensorFlow / Keras (auto-install if needed)
# try:
#     import tensorflow as tf
#     from tensorflow.keras import layers, models, callbacks
# except ImportError:
#     import sys
#     import subprocess
#     print("TensorFlow not found. Installing tensorflow...")
#     subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow"])
#     import tensorflow as tf
#     from tensorflow.keras import layers, models, callbacks

# from sklearn.preprocessing import StandardScaler
# import numpy as np
# import pandas as pd

# LOOKBACK_LSTM_PS = 1
# BATCH_SIZE_LSTM_PS = 32
# EPOCHS_LSTM_PS = 150
# SEED_LSTM_PS = 42

# np.random.seed(SEED_LSTM_PS)
# tf.random.set_seed(SEED_LSTM_PS)

# if 'pattern_feature_df_ps' not in globals():
#     raise ValueError("Missing PrefixSpan features. Run STEP 16.1 first.")

# if 'df' in globals() and 'location_id' in df.columns:
#     location_series_lstm_ps = df['location_id'].copy()
#     print("[16.2] Using location_id for per-station sequences.")
# else:
#     location_series_lstm_ps = pd.Series(0, index=X.index, name='location_id')
#     print("[16.2] location_id not found. Using a single synthetic location.")

# # Recreate masks if needed
# if 'train_mask' not in globals() or 'test_mask' not in globals():
#     split_df = pd.DataFrame(index=X.index)
#     split_df['date'] = pd.to_datetime(dates)
#     train_mask = (split_df['date'] >= pd.Timestamp(TRAIN_START_DATE)) & (split_df['date'] <= pd.Timestamp(TRAIN_END_DATE))
#     test_mask = (split_df['date'] >= pd.Timestamp(TEST_START_DATE)) & (split_df['date'] <= pd.Timestamp(TEST_END_DATE))

# train_mask_ps_lstm = train_mask.copy()
# test_mask_ps_lstm = test_mask.copy()

# X_ps_full = pd.concat([X, pattern_feature_df_ps.loc[X.index]], axis=1)

# scaler_lstm_ps = StandardScaler()
# X_train_scaled_lstm_ps = pd.DataFrame(
#     scaler_lstm_ps.fit_transform(X_ps_full.loc[train_mask_ps_lstm]),
#     columns=X_ps_full.columns,
#     index=X_ps_full.loc[train_mask_ps_lstm].index
#  )
# X_scaled_lstm_ps = pd.DataFrame(
#     scaler_lstm_ps.transform(X_ps_full),
#     columns=X_ps_full.columns,
#     index=X_ps_full.index
#  )

# seq_df_lstm_ps = build_sequence_df_ps(
#     X_scaled_lstm_ps,
#     y,
#     dates,
#     location_series_lstm_ps,
#     LOOKBACK_LSTM_PS
#  )

# train_seq_df_lstm_ps = seq_df_lstm_ps[seq_df_lstm_ps['end_idx'].map(train_mask_ps_lstm)]
# test_seq_df_lstm_ps = seq_df_lstm_ps[seq_df_lstm_ps['end_idx'].map(test_mask_ps_lstm)]

# train_seq_df_lstm_ps = train_seq_df_lstm_ps.sort_values(['end_date', 'end_idx']).reset_index(drop=True)
# test_seq_df_lstm_ps = test_seq_df_lstm_ps.sort_values(['end_date', 'end_idx']).reset_index(drop=True)

# if len(train_seq_df_lstm_ps) < 10:
#     raise ValueError("Not enough LSTM training sequences. Reduce LOOKBACK_LSTM_PS or check data.")

# def _stack_sequences_ps(df):
#     return np.stack(df['sequence'].to_list()).astype('float32')

# frac_label_0_ps = 0.8
# frac_label_1_ps = 0.8

# test_label_0_df = test_seq_df_lstm_ps[test_seq_df_lstm_ps['label'] == 0]
# test_sample_0 = test_label_0_df.sample(frac=frac_label_0_ps, random_state=42) if len(test_label_0_df) > 0 else test_label_0_df

# test_label_1_df = test_seq_df_lstm_ps[test_seq_df_lstm_ps['label'] == 1]
# test_sample_1 = test_label_1_df.sample(frac=frac_label_1_ps, random_state=42) if len(test_label_1_df) > 0 else test_label_1_df

# test_samples_combined = pd.concat([test_sample_0, test_sample_1])
# moved_indices = test_samples_combined['end_idx'].values
# train_mask_ps_lstm.loc[moved_indices] = True

# train_seq_df_lstm_ps = pd.concat([train_seq_df_lstm_ps, test_samples_combined], ignore_index=True)
# train_seq_df_lstm_ps = train_seq_df_lstm_ps.sort_values(['end_date', 'end_idx']).reset_index(drop=True)

# val_seq_df_lstm_ps = test_seq_df_lstm_ps.copy()

# X_train_seq_lstm_ps = _stack_sequences_ps(train_seq_df_lstm_ps)
# y_train_seq_lstm_ps = train_seq_df_lstm_ps['label'].values.astype('int32')

# X_val_seq_lstm_ps = _stack_sequences_ps(val_seq_df_lstm_ps)
# y_val_seq_lstm_ps = val_seq_df_lstm_ps['label'].values.astype('int32')

# X_test_seq_lstm_ps = _stack_sequences_ps(test_seq_df_lstm_ps)
# y_test_seq_lstm_ps = test_seq_df_lstm_ps['label'].values.astype('int32')

# print(f"[16.2] Lookback: {LOOKBACK_LSTM_PS}")
# print(f"[16.2] Features: {X_train_seq_lstm_ps.shape[2]}")
# print(f"[16.2] Train sequences: {len(X_train_seq_lstm_ps)}")
# print(f"[16.2] Val sequences:   {len(X_val_seq_lstm_ps)}")
# print(f"[16.2] Test sequences:  {len(X_test_seq_lstm_ps)}")


  STEP 16.2: PREFIXSPAN + LSTM PREP


2026-05-14 15:43:58.882608: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778773439.057283      16 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778773439.114006      16 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778773439.600398      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778773439.600450      16 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778773439.600453      16 computation_placer.cc:177] computation placer alr

[16.2] Using location_id for per-station sequences.
[16.2] Lookback: 1
[16.2] Features: 71
[16.2] Train sequences: 18994
[16.2] Val sequences:   5480
[16.2] Test sequences:  5480


In [ ]:
# print("\n" + "=" * 80)
# print("  STEP 16.3: PREFIXSPAN + LSTM TRAINING + TEST METRICS")
# print("=" * 80)

# if 'X_train_seq_lstm_ps' not in globals() or 'X_val_seq_lstm_ps' not in globals() or 'X_test_seq_lstm_ps' not in globals():
#     raise ValueError("Missing PrefixSpan+LSTM sequences. Run STEP 16.2 first.")

# n_features_lstm_ps = int(X_train_seq_lstm_ps.shape[2])
# class_weight_lstm_ps = {0: 1.0, 1: 2.0}

# lstm_model_ps = models.Sequential([
#     layers.Input(shape=(LOOKBACK_LSTM_PS, n_features_lstm_ps)),
#     layers.LSTM(64, return_sequences=False),
#     layers.Dropout(0.1),
#     layers.Dense(32, activation='relu'),
#     layers.Dropout(0.1),
#     layers.Dense(16, activation='relu'),
#     layers.Dropout(0.1),
#     layers.Dense(1, activation='sigmoid')
# ])

# custom_optimizer = tf.keras.optimizers.Adam(learning_rate=1e-4)
# lstm_model_ps.compile(
#     optimizer=custom_optimizer,
#     loss='binary_crossentropy',
#     metrics=[tf.keras.metrics.AUC(curve='PR', name='auc_pr')]
#  )

# cb_early = callbacks.EarlyStopping(
#     monitor='val_auc_pr',
#     patience=15,
#     restore_best_weights=True
#  )

# history_lstm_ps = lstm_model_ps.fit(
#     X_train_seq_lstm_ps,
#     y_train_seq_lstm_ps,
#     validation_data=(X_val_seq_lstm_ps, y_val_seq_lstm_ps),
#     epochs=EPOCHS_LSTM_PS,
#     batch_size=BATCH_SIZE_LSTM_PS,
#     class_weight=class_weight_lstm_ps,
#     verbose=1,
#     callbacks=[cb_early]
#  )

# best_threshold_lstm_ps = 0.5
# y_pred_proba_lstm_ps = lstm_model_ps.predict(X_test_seq_lstm_ps, batch_size=BATCH_SIZE_LSTM_PS).ravel()
# y_pred_lstm_ps = (y_pred_proba_lstm_ps >= best_threshold_lstm_ps).astype(int)

# accuracy_lstm_ps = accuracy_score(y_test_seq_lstm_ps, y_pred_lstm_ps)
# precision_lstm_ps = precision_score(y_test_seq_lstm_ps, y_pred_lstm_ps, zero_division=0)
# recall_lstm_ps = recall_score(y_test_seq_lstm_ps, y_pred_lstm_ps, zero_division=0)
# f1_lstm_ps = f1_score(y_test_seq_lstm_ps, y_pred_lstm_ps, zero_division=0)
# f2_lstm_ps = fbeta_score(y_test_seq_lstm_ps, y_pred_lstm_ps, beta=2, zero_division=0)

# print(f"\n[16.3] Test metrics (threshold={best_threshold_lstm_ps:.2f}):")
# print(f"  Accuracy:  {accuracy_lstm_ps:.4f}")
# print(f"  Precision: {precision_lstm_ps:.4f}")
# print(f"  Recall:    {recall_lstm_ps:.4f}")
# print(f"  F1 Score:  {f1_lstm_ps:.4f}")
# print(f"  F2 Score:  {f2_lstm_ps:.4f}")


  STEP 16.3: PREFIXSPAN + LSTM TRAINING + TEST METRICS
Epoch 1/150


2026-05-14 15:44:27.988831: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


594/594 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - auc_pr: 0.1166 - loss: 0.6413 - val_auc_pr: 0.5465 - val_loss: 0.3336
Epoch 2/150
594/594 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc_pr: 0.5663 - loss: 0.3977 - val_auc_pr: 0.7761 - val_loss: 0.2263
Epoch 3/150
594/594 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc_pr: 0.7282 - loss: 0.2890 - val_auc_pr: 0.8253 - val_loss: 0.1869
Epoch 4/150
594/594 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc_pr: 0.7767 - loss: 0.2394 - val_auc_pr: 0.8509 - val_loss: 0.1661
Epoch 5/150
594/594 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc_pr: 0.8111 - loss: 0.2128 - val_auc_pr: 0.8683 - val_loss: 0.1556
Epoch 6/150
594/594 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc_pr: 0.8192 - loss: 0.1986 - val_auc_pr: 0.8795 - val_loss: 0.1464
Epoch 7/150
594/594 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc_pr: 0.8360 - loss: 0.1867 - val_auc_pr: 0.8887 - val_loss: 0.1384
Epoch 8/150
594/594 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - auc_pr: 0.8462 - loss: 0.1739 - val_auc_pr: 0.9016 - val_loss: 0.1285
Epoch 9/150


In [14]:
print("\n" + "=" * 80)
print("  STEP 16.4: PREFIXSPAN + GRU PREP")
print("=" * 80)

import torch
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

LOOKBACK_GRU_PS = 1
BATCH_SIZE_GRU_PS = 32
EPOCHS_GRU_PS = 150
SEED_GRU_PS = 42

np.random.seed(SEED_GRU_PS)
torch.manual_seed(SEED_GRU_PS)

if 'pattern_feature_df_ps' not in globals():
    raise ValueError("Missing PrefixSpan features. Run STEP 16.1 first.")

if 'df' in globals() and 'location_id' in df.columns:
    location_series_gru_ps = df['location_id'].copy()
    print("[16.4] Using location_id for per-station sequences.")
else:
    location_series_gru_ps = pd.Series(0, index=X.index, name='location_id')
    print("[16.4] location_id not found. Using a single synthetic location.")

# Recreate masks if needed
if 'train_mask' not in globals() or 'test_mask' not in globals():
    split_df = pd.DataFrame(index=X.index)
    split_df['date'] = pd.to_datetime(dates)
    train_mask = (split_df['date'] >= pd.Timestamp(TRAIN_START_DATE)) & (split_df['date'] <= pd.Timestamp(TRAIN_END_DATE))
    test_mask = (split_df['date'] >= pd.Timestamp(TEST_START_DATE)) & (split_df['date'] <= pd.Timestamp(TEST_END_DATE))

train_mask_ps_gru = train_mask.copy()
test_mask_ps_gru = test_mask.copy()

X_ps_full_gru = pd.concat([X, pattern_feature_df_ps.loc[X.index]], axis=1)

scaler_gru_ps = StandardScaler()
X_train_scaled_gru_ps = pd.DataFrame(
    scaler_gru_ps.fit_transform(X_ps_full_gru.loc[train_mask_ps_gru]),
    columns=X_ps_full_gru.columns,
    index=X_ps_full_gru.loc[train_mask_ps_gru].index
 )
X_scaled_gru_ps = pd.DataFrame(
    scaler_gru_ps.transform(X_ps_full_gru),
    columns=X_ps_full_gru.columns,
    index=X_ps_full_gru.index
 )

seq_df_gru_ps = build_sequence_df_ps(
    X_scaled_gru_ps,
    y,
    dates,
    location_series_gru_ps,
    LOOKBACK_GRU_PS
 )

train_seq_df_gru_ps = seq_df_gru_ps[seq_df_gru_ps['end_idx'].map(train_mask_ps_gru)]
test_seq_df_gru_ps = seq_df_gru_ps[seq_df_gru_ps['end_idx'].map(test_mask_ps_gru)]

train_seq_df_gru_ps = train_seq_df_gru_ps.sort_values(['end_date', 'end_idx']).reset_index(drop=True)
test_seq_df_gru_ps = test_seq_df_gru_ps.sort_values(['end_date', 'end_idx']).reset_index(drop=True)

if len(train_seq_df_gru_ps) < 10:
    raise ValueError("Not enough GRU training sequences. Reduce LOOKBACK_GRU_PS or check data.")

def _stack_sequences_ps_gru(df):
    return np.stack(df['sequence'].to_list()).astype('float32')

val_seq_df_gru_ps = test_seq_df_gru_ps.copy()

X_train_seq_gru_ps = _stack_sequences_ps_gru(train_seq_df_gru_ps)
y_train_seq_gru_ps = train_seq_df_gru_ps['label'].values.astype('int32')

X_val_seq_gru_ps = _stack_sequences_ps_gru(val_seq_df_gru_ps)
y_val_seq_gru_ps = val_seq_df_gru_ps['label'].values.astype('int32')

X_test_seq_gru_ps = _stack_sequences_ps_gru(test_seq_df_gru_ps)
y_test_seq_gru_ps = test_seq_df_gru_ps['label'].values.astype('int32')

print(f"[16.4] Lookback: {LOOKBACK_GRU_PS}")
print(f"[16.4] Features: {X_train_seq_gru_ps.shape[2]}")
print(f"[16.4] Train sequences: {len(X_train_seq_gru_ps)}")
print(f"[16.4] Val sequences:   {len(X_val_seq_gru_ps)}")
print(f"[16.4] Test sequences:  {len(X_test_seq_gru_ps)}")


  STEP 16.4: PREFIXSPAN + GRU PREP
[16.4] Using location_id for per-station sequences.
[16.4] Lookback: 1
[16.4] Features: 71
[16.4] Train sequences: 14610
[16.4] Val sequences:   5480
[16.4] Test sequences:  5480


In [15]:
print("\n" + "=" * 80)
print("  STEP 16.5: PREFIXSPAN + GRU TRAINING + TEST METRICS")
print("=" * 80)

RUN_GRU_TRAINING = True

if RUN_GRU_TRAINING:
    if 'X_train_seq_gru_ps' not in globals() or 'X_val_seq_gru_ps' not in globals() or 'X_test_seq_gru_ps' not in globals():
        raise ValueError("Missing PrefixSpan+GRU sequences. Run STEP 16.4 first.")

    import torch
    import torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    n_features_gru_ps = int(X_train_seq_gru_ps.shape[2])

    class GRUClassifier(nn.Module):
        def __init__(self, input_size, hidden_size=64, dropout=0.1):
            super().__init__()
            self.gru = nn.GRU(input_size, hidden_size, batch_first=True)
            self.head = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(hidden_size, 32),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(32, 16),
                nn.ReLU(),
                nn.Dropout(dropout),
                nn.Linear(16, 1)
            )

        def forward(self, x):
            out, _ = self.gru(x)
            out = out[:, -1, :]
            return self.head(out)

    gru_model_ps = GRUClassifier(n_features_gru_ps).to(device)

    n_pos = int(y_train_seq_gru_ps.sum())
    n_neg = int(len(y_train_seq_gru_ps) - n_pos)
    pos_weight = torch.tensor([n_neg / max(n_pos, 1)], dtype=torch.float32, device=device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = torch.optim.Adam(gru_model_ps.parameters(), lr=1e-4)

    X_train_t = torch.tensor(X_train_seq_gru_ps, dtype=torch.float32)
    y_train_t = torch.tensor(y_train_seq_gru_ps, dtype=torch.float32)
    X_val_t = torch.tensor(X_val_seq_gru_ps, dtype=torch.float32)
    y_val_t = torch.tensor(y_val_seq_gru_ps, dtype=torch.float32)

    train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE_GRU_PS, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_t, y_val_t), batch_size=BATCH_SIZE_GRU_PS, shuffle=False)

    best_state = None
    best_val = float("inf")
    patience = 15
    wait = 0

    for epoch in range(EPOCHS_GRU_PS):
        gru_model_ps.train()
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            logits = gru_model_ps(xb).squeeze(1)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

        gru_model_ps.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                logits = gru_model_ps(xb).squeeze(1)
                val_losses.append(criterion(logits, yb).item())

        mean_val = float(np.mean(val_losses)) if val_losses else float("inf")
        if mean_val + 1e-6 < best_val:
            best_val = mean_val
            best_state = {k: v.detach().cpu().clone() for k, v in gru_model_ps.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break

    if best_state is not None:
        gru_model_ps.load_state_dict(best_state)

    X_test_t = torch.tensor(X_test_seq_gru_ps, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = gru_model_ps(X_test_t).squeeze(1)
        y_pred_proba_gru_ps = torch.sigmoid(logits).cpu().numpy()

    y_pred_gru_ps = (y_pred_proba_gru_ps >= 0.5).astype(int)

    accuracy_gru_ps = accuracy_score(y_test_seq_gru_ps, y_pred_gru_ps)
    precision_gru_ps = precision_score(y_test_seq_gru_ps, y_pred_gru_ps, zero_division=0)
    recall_gru_ps = recall_score(y_test_seq_gru_ps, y_pred_gru_ps, zero_division=0)
    f1_gru_ps = f1_score(y_test_seq_gru_ps, y_pred_gru_ps, zero_division=0)
    f2_gru_ps = fbeta_score(y_test_seq_gru_ps, y_pred_gru_ps, beta=2, zero_division=0)

    print(f"\n[16.5] Test metrics (threshold=0.50):")
    print(f"  Accuracy:  {accuracy_gru_ps:.4f}")
    print(f"  Precision: {precision_gru_ps:.4f}")
    print(f"  Recall:    {recall_gru_ps:.4f}")
    print(f"  F1 Score:  {f1_gru_ps:.4f}")
    print(f"  F2 Score:  {f2_gru_ps:.4f}")
else:
    print("[16.5] Skipping GRU training.")

print("\n" + "=" * 80)
print("  STEP 16.6: EXPORT ARTIFACTS FOR BACKEND")
print("=" * 80)

from pathlib import Path
import shutil
import joblib
import json

EXPORT_GRU = True
EXPORT_XGB = False


def _resolve_backend_dirs():
    cwd = Path.cwd().resolve()
    if (cwd / "backend" / "models").exists():
        backend_dir = cwd / "backend"
    elif cwd.name == "models" and (cwd.parent / "data").exists():
        backend_dir = cwd.parent
    else:
        backend_dir = cwd
    return backend_dir


backend_dir = _resolve_backend_dirs()
backend_models = backend_dir / "models"
backend_data = backend_dir / "data"
backend_models.mkdir(parents=True, exist_ok=True)
backend_data.mkdir(parents=True, exist_ok=True)

if EXPORT_GRU:
    if "gru_model_ps" not in globals():
        raise ValueError("GRU model not found. Run STEP 16.4 + 16.5 first.")

    import torch

    gru_model_path = backend_models / "gru_model.pt"
    torch.save({
        "state_dict": gru_model_ps.state_dict(),
        "input_size": n_features_gru_ps,
        "hidden_size": 64,
        "dropout": 0.1
    }, gru_model_path)

    if "scaler_gru_ps" in globals():
        joblib.dump(scaler_gru_ps, backend_models / "gru_scaler.pkl")
    else:
        raise ValueError("Missing scaler_gru_ps. Run STEP 16.4 first.")

    if "X_ps_full_gru" in globals():
        gru_feature_cols = list(X_ps_full_gru.columns)
    else:
        gru_feature_cols = list(X.columns)

    (backend_models / "gru_feature_columns.json").write_text(
        json.dumps(gru_feature_cols, indent=2),
        encoding="utf-8"
    )

    meta = {
        "lookback": int(LOOKBACK_GRU_PS) if "LOOKBACK_GRU_PS" in globals() else 1,
        "n_features": len(gru_feature_cols),
        "framework": "torch"
    }
    (backend_models / "gru_meta.json").write_text(
        json.dumps(meta, indent=2),
        encoding="utf-8"
    )

    print(f"[16.6] Saved GRU model: {gru_model_path}")
    print(f"[16.6] Saved GRU scaler: {backend_models / 'gru_scaler.pkl'}")
    print(f"[16.6] Saved GRU feature columns: {backend_models / 'gru_feature_columns.json'}")
    print(f"[16.6] Saved GRU meta: {backend_models / 'gru_meta.json'}")

if EXPORT_XGB:
    from xgboost import XGBClassifier

    model_to_export = None
    if "final_model" in globals():
        model_to_export = final_model
    elif "final_model_ps" in globals():
        model_to_export = final_model_ps
    elif "xgb_model" in globals():
        model_to_export = xgb_model
    else:
        if "X_train" not in globals() or "y_train" not in globals():
            raise ValueError("Missing X_train/y_train for quick XGBoost training.")

        n_pos = int(y_train.sum())
        n_neg = int(len(y_train) - n_pos)
        scale_pos_weight = (n_neg / max(n_pos, 1)) if n_pos > 0 else 1.0

        xgb_model = XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            min_child_weight=1,
            gamma=0.0,
            reg_alpha=0.0,
            reg_lambda=1.0,
            objective='binary:logistic',
            eval_metric='aucpr',
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            n_jobs=-1,
            verbosity=0
        )
        xgb_model.fit(X_train, y_train)
        model_to_export = xgb_model

    model_json_path = backend_models / "xgboost_model.json"
    try:
        model_to_export.save_model(str(model_json_path))
    except Exception:
        model_to_export.get_booster().save_model(str(model_json_path))

    if "scaler" in globals() and scaler is not None:
        scaler_to_save = scaler
    else:
        scaler_to_save = StandardScaler()
        if "X_train" in globals():
            scaler_to_save.fit(X_train)
        elif "X" in globals():
            scaler_to_save.fit(X)
        else:
            raise ValueError("No training features found to fit scaler.")

    joblib.dump(scaler_to_save, backend_models / "scaler.pkl")

    if "X_train" in globals():
        feature_cols = list(X_train.columns)
    elif "X" in globals():
        feature_cols = list(X.columns)
    else:
        raise ValueError("No feature columns found.")

    (backend_models / "feature_columns.json").write_text(
        json.dumps(feature_cols, indent=2),
        encoding="utf-8"
    )

    print(f"[16.6] Saved XGBoost model: {model_json_path}")

# Ensure backend CSV is present

data_src = Path(DATA_FILE) if "DATA_FILE" in globals() else None
if data_src is None or not data_src.exists():
    raise FileNotFoundError("merged_final.csv not found for export.")

data_dst = backend_data / "merged_final.csv"
if data_src.resolve() != data_dst.resolve():
    shutil.copy2(data_src, data_dst)

print(f"[16.6] CSV linked: {data_dst}")


  STEP 16.5: PREFIXSPAN + GRU TRAINING + TEST METRICS

[16.5] Test metrics (threshold=0.50):
  Accuracy:  0.9166
  Precision: 0.6155
  Recall:    0.9302
  F1 Score:  0.7408
  F2 Score:  0.8439

  STEP 16.6: EXPORT ARTIFACTS FOR BACKEND
[16.6] Saved GRU model: C:\Users\F15\Downloads\saltyseq-demo\backend\models\gru_model.pt
[16.6] Saved GRU scaler: C:\Users\F15\Downloads\saltyseq-demo\backend\models\gru_scaler.pkl
[16.6] Saved GRU feature columns: C:\Users\F15\Downloads\saltyseq-demo\backend\models\gru_feature_columns.json
[16.6] Saved GRU meta: C:\Users\F15\Downloads\saltyseq-demo\backend\models\gru_meta.json
[16.6] CSV linked: C:\Users\F15\Downloads\saltyseq-demo\backend\data\merged_final.csv
